# 01. Construcción de la base integrada - Fashion Transparency Index 2023

### Objetivo del notebook

Construir una base de datos a partir de las descargas parciales del **Fashion Transparency Index 2023** realizadas desde WikiRate.

### Descripción general

Debido a que la plataforma limita la exportación a un máximo de 5,000 observaciones por descarga, inicialmente la información se recuperó mediante filtros temáticos disponibles en WikiRate, organizando los archivos descargados en las carpetas Environment, Social y Governance dentro de `raw/`. Posteriormente, se identificó que esta estrategia no recuperaba la totalidad de las métricas del índice, por lo que las métricas restantes fueron descargadas en bloques adicionales y almacenadas en la carpeta `raw/Other`.

Finalmente, todos los archivos fueron verificados, integrados y depurados para obtener una única base de datos consolidada, comprobando la consistencia de las variables, la estructura de los registros y la ausencia de observaciones duplicadas. Esta base constituirá el punto de partida para las etapas posteriores de limpieza, selección de variables y análisis exploratorio de datos.

In [1]:
# Importamos las librerías necesarias

import pandas as pd
from pathlib import Path


# Configuración para visualizar mejor las tablas
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


# Definimos las rutas principales del proyecto
ruta_proyecto = Path(
    "/Users/avrilsalazar/Documents/FashionTransparency2023"
)

ruta_datos = ruta_proyecto / "raw"
ruta_other = ruta_datos / "Other"
ruta_auxiliar = ruta_datos / "Auxiliary"
ruta_procesados = ruta_proyecto / "processed"


# Verificamos que las carpetas principales existan
if not ruta_datos.exists():
    raise FileNotFoundError(
        f"No se encontró la carpeta de datos: {ruta_datos}"
    )

if not ruta_auxiliar.exists():
    raise FileNotFoundError(
        f"No se encontró la carpeta auxiliar: {ruta_auxiliar}"
    )

ruta_procesados.mkdir(
    parents=True,
    exist_ok=True
)

print("Rutas del proyecto configuradas correctamente.")

Rutas del proyecto configuradas correctamente.


### Verificación de la estructura de los archivos

Antes de integrar la información, se verificó que todos los archivos descargados compartieran la misma estructura de variables. Esta comprobación permitió confirmar que la integración de las descargas temáticas podía realizarse de forma consistente mediante la concatenación de los registros.

In [2]:
# Localizamos los archivos correspondientes a las descargas temáticas

carpetas = [
    "Environment",
    "Social",
    "Governance"
]

archivos_csv = []

for carpeta in carpetas:

    archivos_csv.extend(
        sorted(
            (ruta_datos / carpeta).glob("*.csv")
        )
    )

# Verificamos cuántos archivos fueron encontrados
print(
    f"Archivos temáticos encontrados: {len(archivos_csv)}"
)

for archivo in archivos_csv:
    print(archivo.name)

Archivos temáticos encontrados: 18
Biodiversity.csv
Circular_Economy.csv
Climate_Change.csv
Energy.csv
Sustainable_Supply_Chains.csv
Waste.csv
Water.csv
Community_Impact.csv
Diversity_Equity_Inclusion.csv
Human_Rights.csv
Just_Transition.csv
Labor_Rights_Working_Conditions.csv
Corporate_Governance.csv
Financial_Governance.csv
Risk_Management.csv
Sustainability_Strategy.csv
Tax_Transparency.csv
Transparency.csv


In [3]:
# Seleccionamos el primer archivo de la lista para revisar su estructura

archivo_ejemplo = archivos_csv[0]

print(f"Archivo revisado: {archivo_ejemplo}\n")

# Mostramos sus primeras líneas para localizar el encabezado real
with open(archivo_ejemplo, "r", encoding="utf-8-sig") as archivo_texto:
    for numero, linea in enumerate(archivo_texto):
        print(f"{numero}: {linea.strip()}")

        if numero == 9:
            break

Archivo revisado: /Users/avrilsalazar/Documents/FashionTransparency2023/raw/Environment/Biodiversity.csv

0: # https://wikirate.org/Fashion_Transparency_Index_2023_full_dataset+Answer?filter%5Bcompany_keyword%5D=&filter%5Bcompany_identifier%5D%5Btype%5D=&filter%5Bcompany_identifier%5D%5Bvalue%5D=&filter%5Bmetric_keyword%5D=&filter%5Btopic%5D%5B%5D=%7E21475062&filter%5Bvalue%5D=&filter%5Bstatus%5D=exists&export_type=answer&format=csv&view=detailed&limit=5000&utf8=%E2%9C%93
1: "# Wikirate.org, licensed under CC BY 4.0 (https://creativecommons.org/licenses/by/4.0). See https://wikirate.org/Attribution_Guide."
2: # 2026-07-22 21:37:57 UTC
3: #
4: Answer Page,Metric,Company,Year,Value,Source Page,Answer ID,Original Source,Source Count,Comments,ISIN
5: https://wikirate.org/Fashion_Revolution+Fashion_Transparency_Index_2023+Puma+2023,Fashion Revolution+Fashion Transparency Index 2023,Puma,2023,6.6399545215046025,,,,,,DE0006969603;US7458781082;US7458782072
6: https://wikirate.org/Fashion_Revol

In [4]:
# Revisamos la estructura de cada archivo antes de integrarlos

estructura_archivos = []

# Recorremos todos los archivos descargados
for archivo in archivos_csv:

    # Omitimos las primeras 4 líneas informativas
    df = pd.read_csv(archivo, skiprows=4)

    # Guardamos un resumen de la estructura del archivo
    estructura_archivos.append({
        "archivo": archivo.name,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "variables": tuple(df.columns)
    })

# Convertimos el resumen en un DataFrame
estructura = pd.DataFrame(estructura_archivos)

# Mostramos el resumen de cada archivo
estructura[["archivo", "filas", "columnas"]]

,archivo,filas,columnas
0,Biodiversity.csv,750,11
1,Circular_Economy.csv,2000,11
2,Climate_Change.csv,2250,11
3,Energy.csv,1000,11
4,Sustainable_Supply_Chains.csv,2250,11
5,Waste.csv,1250,11
6,Water.csv,1250,11
7,Community_Impact.csv,750,11
8,Diversity_Equity_Inclusion.csv,750,11
9,Human_Rights.csv,1250,11


In [5]:
# Verificamos cuántas estructuras de columnas distintas existen

estructura["variables"].nunique()

1

In [6]:
# Integramos todos los archivos ordenados por categoría en una sola base

lista_dataframes = []

# Recorremos cada archivo descargado
for archivo in archivos_csv:

    # Leemos el archivo omitiendo las líneas informativas
    df = pd.read_csv(archivo, skiprows=4)

    # Registramos la categoría temática del archivo
    df["Categoria"] = archivo.parent.name

    # Agregamos el DataFrame a la lista
    lista_dataframes.append(df)

# Unimos todos los DataFrames
base_integrada = pd.concat(lista_dataframes, ignore_index=True)

# Mostramos el tamaño de la base integrada
print(f"Número total de registros: {len(base_integrada):,}")
print(f"Número de variables: {base_integrada.shape[1]}")

Número total de registros: 27,000
Número de variables: 12


In [7]:
# Revisamos cuántas empresas, indicadores y años contiene la base integrada

print(f"Empresas únicas: {base_integrada['Company'].nunique()}")
print(f"Indicadores únicos: {base_integrada['Metric'].nunique()}")
print(f"Años disponibles: {sorted(base_integrada['Year'].dropna().unique())}")

Empresas únicas: 250
Indicadores únicos: 46
Años disponibles: [2023]


In [8]:
# Obtenemos la lista de métricas presentes en las descargas por categoría

metricas_actuales = sorted(base_integrada["Metric"].dropna().unique())

print(f"Métricas actuales: {len(metricas_actuales)}")

for metrica in metricas_actuales:
    print(metrica)

Métricas actuales: 46
Fashion Revolution+2. Governance Score
Fashion Revolution+4. Know, Show & Fix Score
Fashion Revolution+5. Spotlight Issues Score (2023)
Fashion Revolution+Approach to Defining Sustainable Materials
Fashion Revolution+Decarbonisation Commitment
Fashion Revolution+Decarbonisation Progress
Fashion Revolution+Describes Environmental Due Diligence Process
Fashion Revolution+Discloses Absolute Energy Reduction
Fashion Revolution+Discloses Annual Investment in Decarbonisation
Fashion Revolution+Discloses Breakdown of Reuse/Recyling of Pre-consumer Waste
Fashion Revolution+Discloses Coal Use
Fashion Revolution+Discloses Content of Scope 1, 2 and 3 Emissions
Fashion Revolution+Discloses Efforts to Invest in Supply Chain Workers
Fashion Revolution+Discloses Free on Board (FOB) Price Changes (COVID-19 Response)
Fashion Revolution+Discloses Number of Collective Bargaining Agreements Providing Wages Above Legal Minimum
Fashion Revolution+Discloses Number of Workers Affected by

### Comparación con las métricas del índice

La base integrada contenía información correspondiente a 46 de las 130 métricas del Fashion Transparency Index 2023. Por ello, fue necesario identificar las métricas faltantes para completar la base de datos.

In [9]:
# Registramos la lista completa de métricas del Fashion Transparency Index 2023

texto_metricas = """
Metric
Fashion Revolution+Supply Chain Policies
Fashion Revolution+Supply Chain Policies Align with International Standards
Fashion Revolution+Supply Chain Policies Are Contractual
Fashion Revolution+Supply Chain Policies in Local Language
Fashion Revolution+1.3 Management Procedures
Fashion Revolution+Plan for Improving Human Rights Impacts
Fashion Revolution+Plan for Improving Environmental Impacts
Fashion Revolution+Reports on Efforts to Improve Human Rights Impacts
Fashion Revolution+Reports on Efforts to Improve Environmental Impacts
Fashion Revolution+1.5 Verified Sustainability Report
Fashion Revolution+2.1 Identifies Lead Responsibility for Human Rights & Environmental Issues
Fashion Revolution+Accountable Board Member Identified
Fashion Revolution+Implementation of Board Level Accountability Described
Fashion Revolution+Worker Representation on Board
Fashion Revolution+Responsible Tax Strategy
Fashion Revolution+Employee Incentives to Improve Impacts
Fashion Revolution+Executive Incentives to Improve Impacts
Fashion Revolution+Executive Pay Linked to Environmental and Social Targets
Fashion Revolution+Supplier Incentives to Improve Impacts
Fashion Revolution+3.1 Tier One Factory Disclosure
Fashion Revolution+3.2 Processing Facilities Disclosure
Fashion Revolution+3.3 Raw Materials Suppliers Disclosure
Fashion Revolution+Describes Human Rights Due Diligence Process
Fashion Revolution+Stakeholder Engagement in Human Rights Due Diligence
Fashion Revolution+Approach to Involving Women in Human Rights Due Diligence
Fashion Revolution+Human Rights Risks Impacts and Violations Identified
Fashion Revolution+Prevention Mitigation and Remediation of Human Rights Risks
Fashion Revolution+Human Rights Risk Prevention and Remediation Outcomes Published
Fashion Revolution+Describes Environmental Due Diligence Process
Fashion Revolution+Stakeholder Engagement in Environmental Due Diligence
Fashion Revolution+Environmental Risks Impacts and Violations Identified
Fashion Revolution+Prevention Mitigation and Remediation of Environmental Risks
Fashion Revolution+Environmental Risk Prevention and Remediation Outcomes Published
Fashion Revolution+Scope, Process and Accreditation for Environmental Audits
Fashion Revolution+New Production Facility Criteria
Fashion Revolution+Number or % of off-site worker interviews
Fashion Revolution+Percentage of Audits including Trade Union Representative
Fashion Revolution+Summary of Assessment Findings
Fashion Revolution+Ratings by Named Facilities
Fashion Revolution+Selected Audit Findings by Named Facilities
Fashion Revolution+Full Audit Reports by Named Facilities
Fashion Revolution+Remediation Process
Fashion Revolution+Affected Stakeholder Engagement in Remediation
Fashion Revolution+Exit Strategy
Fashion Revolution+Grievance Mechanism - Direct Employees
Fashion Revolution+Grievance Mechanism - Supply Chain Workers
Fashion Revolution+Grievance Mechanism Implementation - Supply Chain Workers
Fashion Revolution+Grievance Mechanism Disseminated to Supply Chain Workers
Fashion Revolution+Grievance Mechanism in Supplier Policies
Fashion Revolution+Grievance Reporting - Supply Chain Workers
Fashion Revolution+Discloses Approach to Recruitment Fees
Fashion Revolution+Discloses Number of Workers Affected by Recruitment Fees
Fashion Revolution+Discloses Data on Modern Slavery Prevalence
Fashion Revolution+Discloses Approach to Living Wage
Fashion Revolution+Discloses Strategy to Achieving Living Wage
Fashion Revolution+Discloses Progress toward the payment of a Living Wage to workers in the supply chain
Fashion Revolution+Discloses Living Wage Estimates Used for Benchmarking
Fashion Revolution+Discloses Percentage of Workers Receiving Wage Payments Digitally
Fashion Revolution+Publishes Percentage of Workers Paid Above Minimum Wage
Fashion Revolution+Discloses Percentage of Workers Paid By Piece Rate
Fashion Revolution+Reports on Minimum Wage Paid for Daily / Piece Rate Workers
Fashion Revolution+Discloses Proportion of Workers Paid Minimum Wage
Fashion Revolution+Publishes Percentage or Number of Workers Earning a Living Wage
Fashion Revolution+Protects Labour Costs in Price Negotiations
Fashion Revolution+Discloses Quantity of Orders with Labor Cost Protection
Fashion Revolution+Discloses Free on Board (FOB) Price Changes (COVID-19 Response)
Fashion Revolution+Publishes Standard Supplier Agreement Template
Fashion Revolution+Discloses Policy on Up-Front Supplier Payments
Fashion Revolution+Policy to Pay Supplier Within 60 Days
Fashion Revolution+Discloses Time Taken to Pay Purchase Orders
Fashion Revolution+Discloses Quantity of Orders Changed After Original Agreement
Fashion Revolution+Publishes Supplier Feedback on Purchasing Practices
Fashion Revolution+Discloses number or % of supplier facilities that have independent, democratically elected trade unions
Fashion Revolution+Discloses number or % of Workers covered by Collective Bargaining Agreements
Fashion Revolution+Discloses Number of Collective Bargaining Agreements Providing Wages Above Legal Minimum
Fashion Revolution+Discloses Prevalance of Collective Bargaining Violations
Fashion Revolution+Publishes Gender Pay Gap
Fashion Revolution+Publishes Sex-disaggregated Job Distribution
Fashion Revolution+Publishes Gender-based Labour Violations Data
Fashion Revolution+Discloses Gender Equality Actions in Supplier Facilities
Fashion Revolution+Publishes Ethnicity Pay Gap
Fashion Revolution+Publishes Race-disaggregated Job Distribution
Fashion Revolution+Publishes Racial Equality Actions
Fashion Revolution+Discloses Sourced Fibre Breakdown
Fashion Revolution+Sustainable Materials Strategy
Fashion Revolution+Discloses Progress on Sustainable Materials Strategy
Fashion Revolution+Approach to Defining Sustainable Materials
Fashion Revolution+Targets to Reduce Textiles Derived from Virgin Fossil Fuels
Fashion Revolution+Discloses Progress to Reducing Textiles Derived from Virgin Fossil Fuels
Fashion Revolution+Targets to Reduce Virgin Plastics
Fashion Revolution+Discloses Progress to Reducing Virgin Plastics
Fashion Revolution+Minimizing Impact of Microfibres
Fashion Revolution+Discloses Quantity of Products Produced
Fashion Revolution+Commitment to Degrowth
Fashion Revolution+Discloses Quantity of Pre-Production Waste Generated
Fashion Revolution+Discloses Quantity of Post-production Waste Generated
Fashion Revolution+Discloses Breakdown of Reuse/Recyling of Pre-consumer Waste
Fashion Revolution+Discloses Quantity of Products Destroyed
Fashion Revolution+Offers Take-back Schemes
Fashion Revolution+Discloses Take-back Scheme Outcomes
Fashion Revolution+Offers Clothing Longevity Business Models
Fashion Revolution+Offers Repair Services
Fashion Revolution+Discloses Evidence of Developing Circular Solutions
Fashion Revolution+Discloses % of Circular Products
Fashion Revolution+Discloses Efforts to Invest in Supply Chain Workers
Fashion Revolution+Commitment to Eliminate Hazardous Chemicals
Fashion Revolution+Discloses Progress to Eliminate Hazardous Chemicals
Fashion Revolution+Publishes Supplier Wastewater Test Results
Fashion Revolution+Discloses Water Use
Fashion Revolution+Discloses Water-Related Risk Assessment Process
Fashion Revolution+Decarbonisation Commitment
Fashion Revolution+Science Based Targets
Fashion Revolution+Decarbonisation Progress
Fashion Revolution+Discloses Annual Investment in Decarbonisation
Fashion Revolution+Discloses Content of Scope 1, 2 and 3 Emissions
Fashion Revolution+Environmental Profit and Loss Statement
Fashion Revolution+Zero Deforestation Commitment
Fashion Revolution+Zero Deforestation Progress
Fashion Revolution+Implementation of Regenerative Farming Practices
Fashion Revolution+Discloses Absolute Energy Reduction
Fashion Revolution+Discloses Renewable Energy Use
Fashion Revolution+Discloses Coal Use
Fashion Revolution+1.1 Own Operations Policies
Fashion Revolution+Publishes Responsible Purchasing Code of Conduct
Fashion Revolution+1. Policy & Commitments Score
Fashion Revolution+2. Governance Score
Fashion Revolution+4. Know, Show & Fix Score
Fashion Revolution+3. Traceability Score (2023)
Fashion Revolution+5. Spotlight Issues Score (2023)
Fashion Revolution+Fashion Transparency Index 2023
"""

# Convertimos el texto en una lista limpia
metricas_completas = [
    linea.strip()
    for linea in texto_metricas.splitlines()
    if linea.strip() and linea.strip() != "Metric"
]

# Identificamos las métricas que todavía no están en nuestra base
conjunto_actuales = set(metricas_actuales)

metricas_faltantes = [
    metrica
    for metrica in metricas_completas
    if metrica not in conjunto_actuales
]

print(f"Métricas completas: {len(metricas_completas)}")
print(f"Métricas actuales: {len(metricas_actuales)}")
print(f"Métricas faltantes: {len(metricas_faltantes)}")

Métricas completas: 130
Métricas actuales: 46
Métricas faltantes: 84


In [10]:
# Dividimos las métricas faltantes en bloques de máximo 20 (5,000 registros por bloque)

bloques_metricas = [
    metricas_faltantes[i:i + 20]
    for i in range(0, len(metricas_faltantes), 20)
]

# Mostramos el contenido de cada bloque
for numero, bloque in enumerate(bloques_metricas, start=1):
    print(f"\nBLOQUE {numero}: {len(bloque)} métricas")
    
    for metrica in bloque:
        print(metrica)


BLOQUE 1: 20 métricas
Fashion Revolution+Supply Chain Policies
Fashion Revolution+Supply Chain Policies Align with International Standards
Fashion Revolution+Supply Chain Policies Are Contractual
Fashion Revolution+Supply Chain Policies in Local Language
Fashion Revolution+1.3 Management Procedures
Fashion Revolution+Plan for Improving Human Rights Impacts
Fashion Revolution+Plan for Improving Environmental Impacts
Fashion Revolution+Reports on Efforts to Improve Human Rights Impacts
Fashion Revolution+Reports on Efforts to Improve Environmental Impacts
Fashion Revolution+1.5 Verified Sustainability Report
Fashion Revolution+2.1 Identifies Lead Responsibility for Human Rights & Environmental Issues
Fashion Revolution+Accountable Board Member Identified
Fashion Revolution+Implementation of Board Level Accountability Described
Fashion Revolution+Employee Incentives to Improve Impacts
Fashion Revolution+Executive Incentives to Improve Impacts
Fashion Revolution+Supplier Incentives to Imp

### Descarga de métricas faltantes

La comparación realizada mostró que las descargas realizadas mediante filtros temáticos incluían 46 de las 130 métricas del Fashion Transparency Index 2023.

Para recuperar las 84 métricas restantes, estas se dividieron en cinco bloques de descarga. Los primeros cuatro bloques contienen 20 métricas cada uno, equivalentes a 5,000 observaciones por archivo, mientras que el último bloque contiene 4 métricas, equivalentes a 1,000 observaciones.

Los cinco archivos fueron almacenados en la carpeta `raw/Other` para posteriormente verificar su estructura e integrarlos con las descargas temáticas.

In [11]:
# Localizamos los bloques adicionales de métricas

if not ruta_other.exists():
    raise FileNotFoundError(
        f"No se encontró la carpeta de métricas adicionales: {ruta_other}"
    )

archivos_other = sorted(
    ruta_other.glob("*.csv")
)

# Verificamos que se encuentren los cinco bloques esperados
if len(archivos_other) != 5:
    raise ValueError(
        "Se esperaban 5 archivos CSV en la carpeta Other, "
        f"pero se encontraron {len(archivos_other)}."
    )

print(f"Se encontraron {len(archivos_other)} archivos CSV.\n")

for archivo in archivos_other:
    print(archivo.name)

Se encontraron 5 archivos CSV.

Bloque_1.csv
Bloque_2.csv
Bloque_3.csv
Bloque_4.csv
Bloque_5.csv


In [12]:
# Revisamos la estructura de los archivos adicionales

estructura_other = []

# Recorremos cada archivo
for archivo in archivos_other:

    # Leemos el archivo
    df = pd.read_csv(archivo, skiprows=4)

    # Guardamos un resumen de su estructura
    estructura_other.append({
        "archivo": archivo.name,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "variables": tuple(df.columns)
    })

# Convertimos el resumen en un DataFrame
estructura_other = pd.DataFrame(estructura_other)

# Mostramos el resumen
estructura_other[["archivo", "filas", "columnas"]]

,archivo,filas,columnas
0,Bloque_1.csv,5000,6
1,Bloque_2.csv,5000,6
2,Bloque_3.csv,5000,6
3,Bloque_4.csv,5000,6
4,Bloque_5.csv,1000,6


In [13]:
# Comparamos las columnas presentes en cada archivo adicional

columnas_other = pd.DataFrame({
    archivo.name: pd.read_csv(archivo, skiprows=4).columns
    for archivo in archivos_other
})

columnas_other

,Bloque_1.csv,Bloque_2.csv,Bloque_3.csv,Bloque_4.csv,Bloque_5.csv
0,Answer Page,Answer Page,Answer Page,Answer Page,Answer Page
1,Metric,Metric,Metric,Metric,Metric
2,Company,Company,Company,Company,Company
3,Year,Year,Year,Year,Year
4,Value,Value,Value,Value,Value
5,Source Page,Source Page,Source Page,Source Page,Source Page


In [14]:
# Verificamos si las columnas de los archivos adicionales son un subconjunto
# de las columnas de la base anterior

# Obtenemos las columnas de la base anterior
# sin considerar la columna Categoria, que fue agregada durante la integración
columnas_originales = set(base_integrada.columns) - {"Categoria"}

# Obtenemos las columnas de un archivo adicional
columnas_adicionales = set(
    pd.read_csv(archivos_other[0], skiprows=4).columns
)

# Mostramos ambos conjuntos de columnas
print("Columnas originales:", columnas_originales)
print("\nColumnas adicionales:", columnas_adicionales)

# Verificamos si todas las columnas adicionales existen en la base anterior
print(
    "\n¿Las columnas adicionales están contenidas en las originales?",
    columnas_adicionales.issubset(columnas_originales)
)

Columnas originales: {'Original Source', 'Metric', 'Company', 'Year', 'ISIN', 'Answer Page', 'Source Page', 'Source Count', 'Answer ID', 'Comments', 'Value'}

Columnas adicionales: {'Metric', 'Company', 'Year', 'Answer Page', 'Source Page', 'Value'}

¿Las columnas adicionales están contenidas en las originales? True


In [15]:
# Igualamos la estructura de los archivos adicionales

# Guardamos el orden de columnas de la base temática
columnas_base = base_integrada.columns.tolist()

# Creamos una lista para almacenar los DataFrames
lista_other = []

# Recorremos cada archivo adicional
for archivo in archivos_other:

    # Leemos el archivo
    df = pd.read_csv(
        archivo,
        skiprows=4
    )

    # Registramos el origen del archivo
    df["Categoria"] = archivo.parent.name

    # Agregamos las columnas faltantes conservando
    # el tipo de dato observado en la base temática
    for columna in columnas_base:

        if columna not in df.columns:
            df[columna] = pd.Series(
                index=df.index,
                dtype=base_integrada[columna].dtype
            )

    # Reordenamos las columnas para que coincidan
    # con la estructura de la base temática
    df = df[columnas_base]

    # Guardamos el DataFrame
    lista_other.append(df)

In [16]:
# Integramos los archivos adicionales en una sola base

base_other = pd.concat(
    lista_other,
    ignore_index=True
)

# Verificamos el tamaño de la base resultante
print(f"Número total de registros: {len(base_other):,}")
print(f"Número de variables: {base_other.shape[1]}")

Número total de registros: 21,000
Número de variables: 12


In [17]:
# Verificamos registros únicos y duplicados en ambas bases

clave = ["Company", "Metric", "Year"]

print("Base temática")
print("Registros totales:", len(base_integrada))
print("Registros únicos:", len(base_integrada.drop_duplicates(subset=clave)))
print("Duplicados:", base_integrada.duplicated(subset=clave).sum())

print("\nMétricas faltantes")
print("Registros totales:", len(base_other))
print("Registros únicos:", len(base_other.drop_duplicates(subset=clave)))
print("Duplicados:", base_other.duplicated(subset=clave).sum())

Base temática
Registros totales: 27000
Registros únicos: 11500
Duplicados: 15500

Métricas faltantes
Registros totales: 21000
Registros únicos: 21000
Duplicados: 0


### Integración de la base de datos

Una vez homologada la estructura de los archivos adicionales, ambas bases fueron integradas en una sola tabla. Posteriormente, se eliminaron los registros duplicados utilizando como unidad de análisis la combinación **empresa – indicador – año**, obteniendo la versión consolidada del Fashion Transparency Index 2023.

In [18]:
# Integramos la base temática con las métricas adicionales

base_final = pd.concat(
    [base_integrada, base_other],
    ignore_index=True
)

# Eliminamos registros duplicados utilizando como unidad
# de análisis la combinación empresa, indicador y año
base_final = (
    base_final
    .drop_duplicates(subset=clave)
    .reset_index(drop=True)
)

# Validamos la estructura consolidada antes de continuar
if len(base_final) != 32500:
    raise ValueError(
        "La base consolidada no contiene los "
        "32,500 registros esperados."
    )

if base_final["Company"].nunique() != 250:
    raise ValueError(
        "La base consolidada no contiene las "
        "250 empresas esperadas."
    )

if base_final["Metric"].nunique() != 130:
    raise ValueError(
        "La base consolidada no contiene los "
        "130 indicadores esperados."
    )

if base_final.duplicated(subset=clave).any():
    raise ValueError(
        "Todavía existen registros duplicados en la base."
    )

print(f"Registros finales: {len(base_final):,}")
print(f"Variables: {base_final.shape[1]}")
print(f"Empresas únicas: {base_final['Company'].nunique()}")
print(f"Indicadores únicos: {base_final['Metric'].nunique()}")
print(
    "Registros duplicados:",
    base_final.duplicated(subset=clave).sum()
)

Registros finales: 32,500
Variables: 12
Empresas únicas: 250
Indicadores únicos: 130
Registros duplicados: 0


### Incorporación de la variable país de sede

Como complemento de la base integrada, se incorporó la variable País de sede, correspondiente al país donde se encuentra la sede corporativa de cada empresa evaluada. Para su construcción se tomó como referencia el apartado A–Z of Brands del Fashion Transparency Index 2023, el cual identifica, cuando corresponde, la empresa matriz o grupo controlador de diversas marcas. En estos casos, el país de sede fue asignado con base en la empresa matriz. Cuando el informe no especificó una empresa matriz, el país se determinó utilizando información oficial de la propia empresa o marca. Para garantizar la trazabilidad del proceso, también se registró la fuente utilizada para asignar cada país.

Para facilitar la incorporación de esta información, se construyó un archivo auxiliar que relaciona el nombre de cada empresa en la base integrada con su correspondiente empresa matriz, el país de sede y la fuente utilizada para verificar dicha información. Este archivo fue posteriormente integrado a la base consolidada mediante la variable Company.

In [19]:
# Carga del archivo auxiliar de empresas
# para futura asignación de su país de sede

# El archivo parte del listado A–Z of Brands del
# Fashion Transparency Index 2023.

archivo_empresas_pais = (
    ruta_auxiliar / "empresas_pais_trabajo.xlsx"
)

if not archivo_empresas_pais.exists():
    raise FileNotFoundError(
        "No se encontró el archivo auxiliar: "
        f"{archivo_empresas_pais}"
    )

empresas_pais = pd.read_excel(
    archivo_empresas_pais
)

empresas_pais.head()

,Company,Empresa_oficial,Empresa_matriz,Pais_sede,Fuente_pais,Fuente_empresa_matriz
0,AJIO,AJIO,Reliance Retail,NaN,NaN,"Fashion Transparency Index 2023, p. 35"
1,ALDO,ALDO,The Aldo Group Inc.,NaN,NaN,"Fashion Transparency Index 2023, p. 35"
2,Abercrombie & Fitch,Abercrombie & Fitch,Ambercrombie & Fitch,NaN,NaN,"Fashion Transparency Index 2023, p. 35"
3,Adidas AG,Adidas,Adidas AG,NaN,NaN,"Fashion Transparency Index 2023, p. 35"
4,Aeropostale Inc.,Aeropostale,Authentic Brands Group LLC,NaN,NaN,"Fashion Transparency Index 2023, p. 35"


In [20]:
# Revisión de la estructura del archivo auxiliar

# Las columnas vacías se convierten a texto antes de
# incorporar los países y sus respectivas fuentes.

empresas_pais["Pais_sede"] = (
    empresas_pais["Pais_sede"]
    .astype("string")
)

empresas_pais["Fuente_pais"] = (
    empresas_pais["Fuente_pais"]
    .astype("string")
)

print(f"Empresas: {len(empresas_pais)}")

empresas_pais.info()

Empresas: 250
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Company                250 non-null    object
 1   Empresa_oficial        250 non-null    object
 2   Empresa_matriz         137 non-null    object
 3   Pais_sede              0 non-null      string
 4   Fuente_pais            0 non-null      string
 5   Fuente_empresa_matriz  250 non-null    object
dtypes: object(4), string(2)
memory usage: 11.8+ KB


In [21]:
# Frecuencia de empresas por empresa matriz

# Este resumen permite identificar grupos corporativos compartidos
# entre varias marcas, facilitando la asignación del país de sede.

matrices = (
    empresas_pais["Empresa_matriz"]
    .value_counts(dropna=True)
    .reset_index()
)

matrices.columns = [
    "Empresa_matriz",
    "Numero_marcas"
]

display(matrices)

,Empresa_matriz,Numero_marcas
0,LVMH,5
1,Authentic Brands Group LLC,5
2,Inditex,5
3,Kering,4
4,VF Corporation,3
...,...,...
88,Seven & i Holdings Co,1
89,"Amazon.com, Inc.",1
90,Calloway Golf Company,1
91,Onward Holdings,1


In [22]:
# Tabla de referencia para todas las empresas matriz

# Esta tabla documenta el país de sede y la fuente oficial
# utilizada para cada empresa matriz identificada en el
# Fashion Transparency Index 2023.
# La información se completará progresivamente por bloques.

matrices_pais = matrices.copy()

matrices_pais["Pais_sede"] = pd.Series(
    pd.NA,
    index=matrices_pais.index,
    dtype="string"
)

matrices_pais["Fuente_pais"] = pd.Series(
    pd.NA,
    index=matrices_pais.index,
    dtype="string"
)

# Revisión de las primeras 30 empresas matriz
display(
    matrices_pais.head(30)
)

,Empresa_matriz,Numero_marcas,Pais_sede,Fuente_pais
0,LVMH,5,<NA>,<NA>
1,Authentic Brands Group LLC,5,<NA>,<NA>
2,Inditex,5,<NA>,<NA>
3,Kering,4,<NA>,<NA>
4,VF Corporation,3,<NA>,<NA>
5,Boardriders,3,<NA>,<NA>
6,URBN,3,<NA>,<NA>
7,"Nike, Inc.",3,<NA>,<NA>
8,Calzedonia Group,3,<NA>,<NA>
9,Gap Inc.,3,<NA>,<NA>


In [23]:
# Incorporación del país de sede para las primeras 10
# empresas matriz más frecuentes

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "LVMH",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Francia",
    "https://www.lvmh.com/en/our-group"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Authentic Brands Group LLC",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://corporate.authentic.com/offices"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Inditex",
    ["Pais_sede", "Fuente_pais"]
] = [
    "España",
    "https://www.inditex.com"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Kering",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Francia",
    "https://www.kering.com"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "VF Corporation",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.vfc.com"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Boardriders",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.boardriders.com"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "URBN",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.urbn.com"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Nike, Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://investors.nike.com"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Calzedonia Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Italia",
    "https://www.calzedoniagroup.com"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Gap Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.gapinc.com"
]

display(matrices_pais.head(10))

,Empresa_matriz,Numero_marcas,Pais_sede,Fuente_pais
0,LVMH,5,Francia,https://www.lvmh.com/en/our-group
1,Authentic Brands Group LLC,5,Estados Unidos,https://corporate.authentic.com/offices
2,Inditex,5,España,https://www.inditex.com
3,Kering,4,Francia,https://www.kering.com
4,VF Corporation,3,Estados Unidos,https://www.vfc.com
5,Boardriders,3,Estados Unidos,https://www.boardriders.com
6,URBN,3,Estados Unidos,https://www.urbn.com
7,"Nike, Inc.",3,Estados Unidos,https://investors.nike.com
8,Calzedonia Group,3,Italia,https://www.calzedoniagroup.com
9,Gap Inc.,3,Estados Unidos,https://www.gapinc.com


In [24]:
# Incorporación del país de sede para las siguientes 10
# empresas matriz más frecuentes

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "OTB Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Italia",
    "https://www.otb.net/en/about"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "HanesBrands Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://investor.hanesbrands.com"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Tapestry, Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.tapestry.com"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Reliance Retail",
    ["Pais_sede", "Fuente_pais"]
] = [
    "India",
    "https://www.relianceretail.com"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Fruit of the Loom",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.fotlinc.com"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Hudson's Bay Company",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Canadá",
    "https://www.hbc.com"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "BESTSELLER",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Dinamarca",
    "https://bestseller.com"
]

# Nota:
# En el archivo auxiliar la empresa aparece como "Westfarmers",
# aunque el nombre corporativo correcto es "Wesfarmers".
# Se mantiene el nombre original para conservar la consistencia
# con la tabla de trabajo.
matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Westfarmers",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Australia",
    "https://www.wesfarmers.com.au"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Landmark Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Emiratos Árabes Unidos",
    "https://www.landmarkgroup.com"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Prada Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Italia",
    "https://www.pradagroup.com"
]

display(matrices_pais.head(20))

,Empresa_matriz,Numero_marcas,Pais_sede,Fuente_pais
0,LVMH,5,Francia,https://www.lvmh.com/en/our-group
1,Authentic Brands Group LLC,5,Estados Unidos,https://corporate.authentic.com/offices
2,Inditex,5,España,https://www.inditex.com
3,Kering,4,Francia,https://www.kering.com
4,VF Corporation,3,Estados Unidos,https://www.vfc.com
5,Boardriders,3,Estados Unidos,https://www.boardriders.com
6,URBN,3,Estados Unidos,https://www.urbn.com
7,"Nike, Inc.",3,Estados Unidos,https://investors.nike.com
8,Calzedonia Group,3,Italia,https://www.calzedoniagroup.com
9,Gap Inc.,3,Estados Unidos,https://www.gapinc.com


In [25]:
# Incorporación del país de sede para las últimas 10
# empresas matriz del primer bloque de 30

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Capri Holdings",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://www.capriholdings.com/resources/investor-faqs/default.aspx"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Fast Retailing",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Japón",
    "https://www.fastretailing.com/eng/about/company/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "PVH",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://careers.pvh.com/north-america"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "boohoo group plc",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://www.boohooplc.com/sites/boohoo-corp/files/2023-05/boohoo-group-plc-annual-report-and-accounts-2023.pdf"
]

# Se utiliza el apóstrofo simple porque así aparece el nombre
# de la empresa matriz en la tabla auxiliar.
matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Macy's Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.macysinc.com/investors/investor-resources/investor-faqs/default.aspx"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Otto Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Alemania",
    "https://www.ottogroup.com/en/careers/kgen/ogh.php"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "ALDI Einkauf GmbH & Co. oHG",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Alemania",
    "https://www.aldi-nord.de/kundeninformationen/impressum.html"
]

# L Brands corresponde al nombre histórico registrado en el
# Fashion Transparency Index 2023. Se conserva sin modificar
# para mantener la consistencia con la tabla auxiliar.
matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "L Brands",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.sec.gov/Archives/edgar/data/701985/000070198520000010/lb21202010k.htm"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Walmart Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://corporate.walmart.com/about/newhomeoffice/faq-new-home-office"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Maus Frères",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Suiza",
    "https://maus.ch/en/contact/"
]

# Revisión de las primeras 30 empresas matriz
display(matrices_pais.head(30))

,Empresa_matriz,Numero_marcas,Pais_sede,Fuente_pais
0,LVMH,5,Francia,https://www.lvmh.com/en/our-group
1,Authentic Brands Group LLC,5,Estados Unidos,https://corporate.authentic.com/offices
2,Inditex,5,España,https://www.inditex.com
3,Kering,4,Francia,https://www.kering.com
4,VF Corporation,3,Estados Unidos,https://www.vfc.com
5,Boardriders,3,Estados Unidos,https://www.boardriders.com
6,URBN,3,Estados Unidos,https://www.urbn.com
7,"Nike, Inc.",3,Estados Unidos,https://investors.nike.com
8,Calzedonia Group,3,Italia,https://www.calzedoniagroup.com
9,Gap Inc.,3,Estados Unidos,https://www.gapinc.com


In [26]:
# Revisión del segundo bloque de empresas matriz (31 a 60)

display(
    matrices_pais
    .iloc[30:60]
    .reset_index(drop=True)
)

,Empresa_matriz,Numero_marcas,Pais_sede,Fuente_pais
0,Stockmann Group,1,<NA>,<NA>
1,Telemos Capital Limited,1,<NA>,<NA>
2,Permira,1,<NA>,<NA>
3,"Wolverine World Wide, Inc.",1,<NA>,<NA>
4,Adidas AG,1,<NA>,<NA>
5,Groupe Casino,1,<NA>,<NA>
6,Ryohin Keikaku Co.,1,<NA>,<NA>
7,Nutmeg,1,<NA>,<NA>
8,Cencosud,1,<NA>,<NA>
9,Associated British Foods plc,1,<NA>,<NA>


In [27]:
# Incorporación del país de sede para las empresas matriz
# ubicadas entre las posiciones 31 y 40

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Stockmann Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Finlandia",
    "https://report.stockmanngroup.com/year2019/pdf/Stockmann_corporate_social_responsibility_2019.pdf"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Telemos Capital Limited",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://telemoscapital.com/about-us/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Permira",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://www.permira.com/contact-us/london"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Wolverine World Wide, Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://investors.wolverineworldwide.com/overview/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Adidas AG",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Alemania",
    "https://www.adidas-group.com/en/about/headquarters"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Groupe Casino",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Francia",
    "https://www.groupe-casino.fr/en/legal-notice/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Ryohin Keikaku Co.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Japón",
    "https://www.ryohin-keikaku.jp/en/corporate/overview"
]

# Nutmeg es una marca de Morrisons. El país de sede se asigna
# con base en la información corporativa oficial de Morrisons.
matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Nutmeg",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://www.morrisons-corporate.com/financial-archive/investor-contacts/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Cencosud",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Chile",
    "https://www.cencosud.com/cencosud/site/docs/20240409/20240409224848/memoria_cencosud_2023__eng__con_declaracion.pdf"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Associated British Foods plc",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://www.abf.co.uk/contact"
]

# Revisión de las empresas matriz ubicadas entre las posiciones 31 y 40
display(
    matrices_pais
    .iloc[30:40]
    .reset_index(drop=True)
)

,Empresa_matriz,Numero_marcas,Pais_sede,Fuente_pais
0,Stockmann Group,1,Finlandia,https://report.stockmanngroup.com/year2019/pdf...
1,Telemos Capital Limited,1,Reino Unido,https://telemoscapital.com/about-us/
2,Permira,1,Reino Unido,https://www.permira.com/contact-us/london
3,"Wolverine World Wide, Inc.",1,Estados Unidos,https://investors.wolverineworldwide.com/overv...
4,Adidas AG,1,Alemania,https://www.adidas-group.com/en/about/headquar...
5,Groupe Casino,1,Francia,https://www.groupe-casino.fr/en/legal-notice/
6,Ryohin Keikaku Co.,1,Japón,https://www.ryohin-keikaku.jp/en/corporate/ove...
7,Nutmeg,1,Reino Unido,https://www.morrisons-corporate.com/financial-...
8,Cencosud,1,Chile,https://www.cencosud.com/cencosud/site/docs/20...
9,Associated British Foods plc,1,Reino Unido,https://www.abf.co.uk/contact


In [28]:
# Incorporación del país de sede para las empresas matriz
# ubicadas entre las posiciones 41 y 50

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "LPP",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Polonia",
    "https://www.lpp.com/en/about-us/headquarters-offices-and-logistics/"
]

# En el listado A–Z del Fashion Transparency Index 2023
# aparece escrito como "Shenzen". Se conserva esta escritura
# para mantener la correspondencia con la tabla auxiliar.
matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Shenzen Globalegrow E-Commerce Co., Ltd.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "China",
    "https://static.cninfo.com.cn/finalpage/2021-06-22/1210302148.PDF"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "S Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Finlandia",
    "https://s-ryhma.fi/en/contact-information/sok-corporation"
]

# Tu Clothing corresponde a una marca de Sainsbury's.
# El país se asigna utilizando la información corporativa
# oficial incluida en sus términos y condiciones.
matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Tu Clothing",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://tuclothing.sainsburys.co.uk/help/terms-and-conditions"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "SMCP",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Francia",
    "https://www.smcp.com/app/uploads/2025/04/smcp-2024-urd-en-mel.pdf"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Semir Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "China",
    "https://www.semir.com/biz/contact"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Galeries Lafayette Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Francia",
    "https://www.galerieslafayette.com/service/mentions-legales"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Future Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "India",
    "https://www.felindia.in/Overview.html"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Pentland Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://pentlandgroup.com/about-us/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Frasers Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://find-and-update.company-information.service.gov.uk/company/06035106"
]

# Revisión de las empresas matriz ubicadas entre
# las posiciones 41 y 50
display(
    matrices_pais
    .iloc[40:50]
    .reset_index(drop=True)
)

,Empresa_matriz,Numero_marcas,Pais_sede,Fuente_pais
0,LPP,1,Polonia,https://www.lpp.com/en/about-us/headquarters-o...
1,"Shenzen Globalegrow E-Commerce Co., Ltd.",1,China,https://static.cninfo.com.cn/finalpage/2021-06...
2,S Group,1,Finlandia,https://s-ryhma.fi/en/contact-information/sok-...
3,Tu Clothing,1,Reino Unido,https://tuclothing.sainsburys.co.uk/help/terms...
4,SMCP,1,Francia,https://www.smcp.com/app/uploads/2025/04/smcp-...
5,Semir Group,1,China,https://www.semir.com/biz/contact
6,Galeries Lafayette Group,1,Francia,https://www.galerieslafayette.com/service/ment...
7,Future Group,1,India,https://www.felindia.in/Overview.html
8,Pentland Group,1,Reino Unido,https://pentlandgroup.com/about-us/
9,Frasers Group,1,Reino Unido,https://find-and-update.company-information.se...


In [29]:
# Incorporación del país de sede para las empresas matriz
# ubicadas entre las posiciones 51 y 60

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "TJX",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.tjx.com/investors/investor-resources/contacts"
]

# En el listado del Fashion Transparency Index 2023 aparece
# escrito como "Ambercrombie". Se conserva esta escritura para
# mantener la correspondencia con la tabla auxiliar.
matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Ambercrombie & Fitch",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://corporate.abercrombie.com/contact-us/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "The Very Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://www.theverygroup.com/contacts/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "The Walt Disney Company",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://thewaltdisneycompany.com/app/uploads/2023/11/TWDC_Bylaws_November_2023.pdf"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Oxford Industries, Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://investor.oxfordinc.com/contact-us"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "AEON",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Japón",
    "https://www.aeon.info/en/company/overview/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Woolworths Holdings Limited",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Sudáfrica",
    "https://www.woolworthsholdings.co.za/investors/equity-investors/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Deckers Brands",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.deckers.com/contact"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Shimamura Co., Ltd.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Japón",
    "https://www.shimamura.gr.jp/en/assets-c/uploads/companyinfo_eng.pdf"
]

# Se utiliza el portal corporativo oficial de Loblaw debido a
# que el reporte anual consultado anteriormente restringía el acceso.
matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Loblaw Companies Limited",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Canadá",
    "https://careers.loblaw.ca/"
]

# Revisión de las empresas matriz ubicadas entre
# las posiciones 51 y 60
display(
    matrices_pais
    .iloc[50:60]
    .reset_index(drop=True)
)

,Empresa_matriz,Numero_marcas,Pais_sede,Fuente_pais
0,TJX,1,Estados Unidos,https://www.tjx.com/investors/investor-resourc...
1,Ambercrombie & Fitch,1,Estados Unidos,https://corporate.abercrombie.com/contact-us/
2,The Very Group,1,Reino Unido,https://www.theverygroup.com/contacts/
3,The Walt Disney Company,1,Estados Unidos,https://thewaltdisneycompany.com/app/uploads/2...
4,"Oxford Industries, Inc.",1,Estados Unidos,https://investor.oxfordinc.com/contact-us
5,AEON,1,Japón,https://www.aeon.info/en/company/overview/
6,Woolworths Holdings Limited,1,Sudáfrica,https://www.woolworthsholdings.co.za/investors...
7,Deckers Brands,1,Estados Unidos,https://www.deckers.com/contact
8,"Shimamura Co., Ltd.",1,Japón,https://www.shimamura.gr.jp/en/assets-c/upload...
9,Loblaw Companies Limited,1,Canadá,https://careers.loblaw.ca/


In [30]:
# Revisión del último bloque de empresas matriz (61 a 93)

display(
    matrices_pais
    .iloc[60:]
    .reset_index(drop=True)
)

,Empresa_matriz,Numero_marcas,Pais_sede,Fuente_pais
0,Sear Holdings,1,<NA>,<NA>
1,Kynetic,1,<NA>,<NA>
2,Puig,1,<NA>,<NA>
3,Vivarte,1,<NA>,<NA>
4,Carter's Inc,1,<NA>,<NA>
5,Richemont,1,<NA>,<NA>
6,Woolworths Group,1,<NA>,<NA>
7,Tendam,1,<NA>,<NA>
8,Cotton On Group,1,<NA>,<NA>
9,G-III Apparel Group,1,<NA>,<NA>


In [31]:
# Incorporación del país de sede para las empresas matriz
# ubicadas entre las posiciones 61 y 70

# En la tabla auxiliar aparece como "Sear Holdings".
# Se conserva esta escritura para mantener la correspondencia,
# aunque el nombre corporativo correcto es Sears Holdings Corporation.

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Sear Holdings",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.sec.gov/Archives/edgar/data/1310067/000119312512114869/d276653d10k.htm"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Kynetic",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.sec.gov/Archives/edgar/data/1634117/000095017024079863/0000950170-24-079863-index.htm"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Puig",
    ["Pais_sede", "Fuente_pais"]
] = [
    "España",
    "https://www.puig.com/es/contact/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Vivarte",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Francia",
    "https://annuaire-entreprises.data.gouv.fr/entreprise/vivarte-services-413157090"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Carter's Inc",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://ir.carters.com/shareholder-services/ir-contact"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Richemont",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Suiza",
    "https://www.richemont.com/richemont-privacy-policy/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Woolworths Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Australia",
    "https://www.woolworthsgroup.com.au/au/en/contact-us.html"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Tendam",
    ["Pais_sede", "Fuente_pais"]
] = [
    "España",
    "https://www.tendam.es/contacto/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Cotton On Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Australia",
    "https://cottonongroup.com.au/help-contact-us/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "G-III Apparel Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.giii.com/pages/contact"
]

# Revisión de las empresas matriz ubicadas entre
# las posiciones 61 y 70
display(
    matrices_pais
    .iloc[60:70]
    .reset_index(drop=True)
)

,Empresa_matriz,Numero_marcas,Pais_sede,Fuente_pais
0,Sear Holdings,1,Estados Unidos,https://www.sec.gov/Archives/edgar/data/131006...
1,Kynetic,1,Estados Unidos,https://www.sec.gov/Archives/edgar/data/163411...
2,Puig,1,España,https://www.puig.com/es/contact/
3,Vivarte,1,Francia,https://annuaire-entreprises.data.gouv.fr/entr...
4,Carter's Inc,1,Estados Unidos,https://ir.carters.com/shareholder-services/ir...
5,Richemont,1,Suiza,https://www.richemont.com/richemont-privacy-po...
6,Woolworths Group,1,Australia,https://www.woolworthsgroup.com.au/au/en/conta...
7,Tendam,1,España,https://www.tendam.es/contacto/
8,Cotton On Group,1,Australia,https://cottonongroup.com.au/help-contact-us/
9,G-III Apparel Group,1,Estados Unidos,https://www.giii.com/pages/contact


In [32]:
# Incorporación del país de sede para las empresas matriz
# ubicadas entre las posiciones 71 y 80

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Designer Brands",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://designerbrands.com/footer/contact-us/"
]

# Para Association Familiale Mulliez se conserva como fuente
# principal el sitio oficial de la organización.
matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Association Familiale Mulliez",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Francia",
    "https://www.afm.family/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Samsung C&T",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Corea del Sur",
    "https://www.samsungcnt.com/eng/about-us/contact-info/location.do"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "VARNER",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Noruega",
    "https://varner.com/en/contact/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Tesco PLC",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://www.tescoplc.com/contacts/general/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Caleres",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.caleres.com/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Fenix Outdoor",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Suiza",
    "https://www.fenixoutdoor.com/wp-content/uploads/2026/03/ANNUAL-REPORT-2025_FINAL.pdf"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Berkshire Hathaway",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.berkshirehathaway.com/2025ar/202510-k.pdf"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "TFG",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Sudáfrica",
    "https://tfglimited.co.za/contact/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Fossil Group, Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.fossilgroup.com/investors/financials/investor-resources/investor-faq/"
]

# Revisión de las empresas matriz ubicadas entre
# las posiciones 71 y 80
display(
    matrices_pais
    .iloc[70:80]
    .reset_index(drop=True)
)

,Empresa_matriz,Numero_marcas,Pais_sede,Fuente_pais
0,Designer Brands,1,Estados Unidos,https://designerbrands.com/footer/contact-us/
1,Association Familiale Mulliez,1,Francia,https://www.afm.family/
2,Samsung C&T,1,Corea del Sur,https://www.samsungcnt.com/eng/about-us/contac...
3,VARNER,1,Noruega,https://varner.com/en/contact/
4,Tesco PLC,1,Reino Unido,https://www.tescoplc.com/contacts/general/
5,Caleres,1,Estados Unidos,https://www.caleres.com/
6,Fenix Outdoor,1,Suiza,https://www.fenixoutdoor.com/wp-content/upload...
7,Berkshire Hathaway,1,Estados Unidos,https://www.berkshirehathaway.com/2025ar/20251...
8,TFG,1,Sudáfrica,https://tfglimited.co.za/contact/
9,"Fossil Group, Inc.",1,Estados Unidos,https://www.fossilgroup.com/investors/financia...


In [33]:
# Incorporación del país de sede para las últimas
# 13 empresas matriz (posiciones 81 a 93)

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "JAB Holding Company",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Luxemburgo",
    "https://www.jabholco.com/wp-content/uploads/2026/02/JAB_Holding_Company_S.ar_.l-Annual_Report-2024.pdf"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "The Aldo Group Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Canadá",
    "https://www.aldogroup.com/en/contact"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "TDR Capital",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://www.tdrcapital.com/contact/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Giorgio Armani S.p.A",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Italia",
    "https://www.armani.com/en/legal/terms-and-condition-use/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "H&M Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Suecia",
    "https://hmgroup.com/about-us/contact-us/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Canadian Tire Corporation",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Canadá",
    "https://corp.canadiantire.ca/canadian-tire-head-office-contact-information/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Abercrombie & Fitch",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://corporate.abercrombie.com/contact-us/"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Marquee Brands",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.marqueebrands.com/contact"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Seven & i Holdings Co",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Japón",
    "https://www.7andi.com/en/company/profile"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Amazon.com, Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.sec.gov/Archives/edgar/data/1018724/000101872426000004/amzn-20251231.htm"
]

# En la tabla auxiliar aparece como "Calloway Golf Company".
# Se conserva esta escritura para mantener la correspondencia,
# aunque el nombre corporativo correcto es Callaway Golf Company.
matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Calloway Golf Company",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.callawaygolf.com/legal"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Onward Holdings",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Japón",
    "https://www.onward-hd.co.jp/en/company/overview.html"
]

matrices_pais.loc[
    matrices_pais["Empresa_matriz"] == "Kontoor",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.kontoorbrands.com/contact"
]

# Revisión de las últimas 13 empresas matriz
display(
    matrices_pais
    .iloc[80:]
    .reset_index(drop=True)
)

,Empresa_matriz,Numero_marcas,Pais_sede,Fuente_pais
0,JAB Holding Company,1,Luxemburgo,https://www.jabholco.com/wp-content/uploads/20...
1,The Aldo Group Inc.,1,Canadá,https://www.aldogroup.com/en/contact
2,TDR Capital,1,Reino Unido,https://www.tdrcapital.com/contact/
3,Giorgio Armani S.p.A,1,Italia,https://www.armani.com/en/legal/terms-and-cond...
4,H&M Group,1,Suecia,https://hmgroup.com/about-us/contact-us/
5,Canadian Tire Corporation,1,Canadá,https://corp.canadiantire.ca/canadian-tire-hea...
6,Abercrombie & Fitch,1,Estados Unidos,https://corporate.abercrombie.com/contact-us/
7,Marquee Brands,1,Estados Unidos,https://www.marqueebrands.com/contact
8,Seven & i Holdings Co,1,Japón,https://www.7andi.com/en/company/profile
9,"Amazon.com, Inc.",1,Estados Unidos,https://www.sec.gov/Archives/edgar/data/101872...


In [34]:
# Validación de la tabla de empresas matriz

print(f"Empresas matriz registradas: {len(matrices_pais)}")
print(f"Países pendientes: {matrices_pais['Pais_sede'].isna().sum()}")
print(f"Fuentes pendientes: {matrices_pais['Fuente_pais'].isna().sum()}")

Empresas matriz registradas: 93
Países pendientes: 0
Fuentes pendientes: 0


In [35]:
# Asignamos país de sede y fuente por empresa matriz

mapa_pais_matriz = matrices_pais.set_index(
    "Empresa_matriz"
)["Pais_sede"]

mapa_fuente_matriz = matrices_pais.set_index(
    "Empresa_matriz"
)["Fuente_pais"]


# Incorporación de la información al archivo auxiliar
tiene_matriz = empresas_pais["Empresa_matriz"].notna()

empresas_pais.loc[tiene_matriz, "Pais_sede"] = (
    empresas_pais.loc[tiene_matriz, "Empresa_matriz"]
    .map(mapa_pais_matriz)
)

empresas_pais.loc[tiene_matriz, "Fuente_pais"] = (
    empresas_pais.loc[tiene_matriz, "Empresa_matriz"]
    .map(mapa_fuente_matriz)
)


# Validación
print(
    "Empresas con país asignado:",
    empresas_pais["Pais_sede"].notna().sum()
)

print(
    "Empresas todavía pendientes:",
    empresas_pais["Pais_sede"].isna().sum()
)

display(empresas_pais.head())

Empresas con país asignado: 137
Empresas todavía pendientes: 113


,Company,Empresa_oficial,Empresa_matriz,Pais_sede,Fuente_pais,Fuente_empresa_matriz
0,AJIO,AJIO,Reliance Retail,India,https://www.relianceretail.com,"Fashion Transparency Index 2023, p. 35"
1,ALDO,ALDO,The Aldo Group Inc.,Canadá,https://www.aldogroup.com/en/contact,"Fashion Transparency Index 2023, p. 35"
2,Abercrombie & Fitch,Abercrombie & Fitch,Ambercrombie & Fitch,Estados Unidos,https://corporate.abercrombie.com/contact-us/,"Fashion Transparency Index 2023, p. 35"
3,Adidas AG,Adidas,Adidas AG,Alemania,https://www.adidas-group.com/en/about/headquar...,"Fashion Transparency Index 2023, p. 35"
4,Aeropostale Inc.,Aeropostale,Authentic Brands Group LLC,Estados Unidos,https://corporate.authentic.com/offices,"Fashion Transparency Index 2023, p. 35"


In [36]:
# Empresas que todavía no tienen país de sede asignado

empresas_pendientes = (
    empresas_pais[
        empresas_pais["Pais_sede"].isna()
    ]
    .reset_index(drop=True)
)

print(f"Empresas pendientes: {len(empresas_pendientes)}")


# Revisión de la primera tanda de empresas pendientes (1 a 38)
display(
    empresas_pendientes
    .iloc[:38]
    .reset_index(drop=True)
)

Empresas pendientes: 113


,Company,Empresa_oficial,Empresa_matriz,Pais_sede,Fuente_pais,Fuente_empresa_matriz
0,American Eagle Outfitters,American Eagle,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
1,Anta Sports Products,ANTA,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
2,Aritzia,Aritzia,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
3,Asics Corporation,ASICS,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
4,Asos,ASOS,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
5,Belle International Holdings,Belle,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
6,Bosideng International Holdings Limited,Bosideng,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
7,Brunello Cucinelli,Brunello Cucinelli,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
8,Buckle Inc,Buckle,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
9,Burberry Group plc,Burberry,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"


In [37]:
# Incorporación del país de sede para la primera tanda
# de 38 empresas sin empresa matriz especificada

empresas_pais.loc[
    empresas_pais["Company"] == "American Eagle Outfitters",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://investors.ae.com/investor-resources/default.aspx"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Anta Sports Products",
    ["Pais_sede", "Fuente_pais"]
] = [
    "China",
    "https://ir.anta.com/en/about_info.php"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Aritzia",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Canadá",
    "https://investors.aritzia.com/investor-resources/faqs/default.aspx"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Asics Corporation",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Japón",
    "https://corp.asics.com/en/about_asics/practical_information"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Asos",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://www.asosplc.com/contact"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Belle International Holdings",
    ["Pais_sede", "Fuente_pais"]
] = [
    "China",
    "https://www.belleintl.com/contact"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Bosideng International Holdings Limited",
    ["Pais_sede", "Fuente_pais"]
] = [
    "China",
    "https://www.bosideng.com/enmobile/support/law.html"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Brunello Cucinelli",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Italia",
    "https://www.brunellocucinelli.com/en/contacts.html"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Buckle Inc",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://corporate.buckle.com/contacts/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Burberry Group plc",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://www.burberryplc.com/contacts"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Burlington Stores Inc",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.burlingtoninvestors.com/ir-resources/contact-ir"
]

empresas_pais.loc[
    empresas_pais["Company"] == "C&A",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Suiza",
    "https://www.c-and-a.com/eu/en/corporate/imprint"
]

empresas_pais.loc[
    empresas_pais["Company"] == "C&J Clark International",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://corporate.clarks.com/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Canada Goose",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Canadá",
    "https://investor.canadagoose.com/resources/investor-faqs/default.aspx"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Carhartt Inc",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.carhartt.com/about"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Carrefour S.A.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Francia",
    "https://www.carrefour.com/en/legal-notices"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Celio International",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Francia",
    "https://www.celio.com/en-fr/legal-notice.html"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Chanel SA",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Francia",
    "https://pr-watch.chanel.com/watchesandwonders2025/legals"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Chico's FAS Inc",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.sec.gov/Archives/edgar/data/897429/000089742923000044/chs-20230128.htm"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Children's Place Inc",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.childrensplace.com/us/help-center/customer-service/investor-public-relations"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Columbia Sportswear",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://investor.columbia.com/company-information/faq"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Costco Wholesale",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://investor.costco.com/company-profile/default.aspx"
]

# Se utiliza la nueva fuente oficial accesible de Deichmann.
empresas_pais.loc[
    empresas_pais["Company"] == "Deichmann SE",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Alemania",
    "https://corpsite.deichmann.com/de-DE/unternehmen/deichmann-campus"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Desigual",
    ["Pais_sede", "Fuente_pais"]
] = [
    "España",
    "https://www.desigual.com/en_US/legal-notice-privacy.html"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Dick's Sporting Goods",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://investors.dicks.com/resources/investor-contacts/default.aspx"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Dillard's, Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://investor.dillards.com/investor-resources/contact-us"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Dolce & Gabbana",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Italia",
    "https://world.dolcegabbana.com/corporate/subsidiaries"
]

empresas_pais.loc[
    empresas_pais["Company"] == "El Corte Ingles S.A.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "España",
    "https://www.elcorteingles.es/informacioncorporativa/en/contact/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Ermenegildo Zegna",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Italia",
    "https://www.zegnagroup.com/en/zegna-careers/locations/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Esprit Holdings Limited",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Hong Kong",
    "https://www.hkexnews.hk/listedco/listconews/sehk/2024/0328/2024032800005.pdf"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Express Inc",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.sec.gov/Archives/edgar/data/1483510/000148351023000024/expr-20230128.htm"
]

empresas_pais.loc[
    empresas_pais["Company"] == "FILA",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Corea del Sur",
    "https://www.filaholdings.com/resource/file/en/YOUR%20FILA%20IMPACT_2022%20%28ENG%29.pdf"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Fabletics",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.fabletics.com/privacy"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Falabella",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Chile",
    "https://www.gtm.falabella.com/falabella-cl/category/cat40004/Informacion-Corporativa"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Fashion Nova",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.fashionnova.com/en-tf/pages/caption-contest"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Foot Locker Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.footlocker-inc.com/content/flinc-aem-site/en/home/contact.html"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Furla",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Italia",
    "https://www.furla.com/sk/en/eshop/company-information"
]

empresas_pais.loc[
    empresas_pais["Company"] == "G-star RAW",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Países Bajos",
    "https://www.g-star.com/en_nl/countries-contactinfo"
]

# Validación de la primera tanda de 38 empresas
print(
    "Empresas todavía pendientes:",
    empresas_pais["Pais_sede"].isna().sum()
)

display(
    empresas_pais[
        empresas_pais["Company"].isin(
            empresas_pendientes.iloc[:38]["Company"]
        )
    ][
        [
            "Company",
            "Pais_sede",
            "Fuente_pais"
        ]
    ].reset_index(drop=True)
)

Empresas todavía pendientes: 75


,Company,Pais_sede,Fuente_pais
0,American Eagle Outfitters,Estados Unidos,https://investors.ae.com/investor-resources/de...
1,Anta Sports Products,China,https://ir.anta.com/en/about_info.php
2,Aritzia,Canadá,https://investors.aritzia.com/investor-resourc...
3,Asics Corporation,Japón,https://corp.asics.com/en/about_asics/practica...
4,Asos,Reino Unido,https://www.asosplc.com/contact
5,Belle International Holdings,China,https://www.belleintl.com/contact
6,Bosideng International Holdings Limited,China,https://www.bosideng.com/enmobile/support/law....
7,Brunello Cucinelli,Italia,https://www.brunellocucinelli.com/en/contacts....
8,Buckle Inc,Estados Unidos,https://corporate.buckle.com/contacts/
9,Burberry Group plc,Reino Unido,https://www.burberryplc.com/contacts


In [38]:
# Revisión de la segunda tanda de empresas pendientes (39 a 76)

display(
    empresas_pendientes
    .iloc[38:76]
    .reset_index(drop=True)
)

,Company,Empresa_oficial,Empresa_matriz,Pais_sede,Fuente_pais,Fuente_empresa_matriz
0,Gerry Weber,Gerry Weber,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
1,Gildan Activewear Inc.,Gildan,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
2,Guess? Inc,GUESS,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
3,Gymshark Limited,Gymshark,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
4,Heilan Home,Heilan Home,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
5,Hema,HEMA,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
6,Hermes International,Hermès,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
7,Hugo Boss AG,Hugo Boss,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
8,JD Sports Fashion plc,JD Sports,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
9,Jockey International Inc,Jockey,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"


In [39]:
# Incorporación del país de sede para la segunda tanda
# de 38 empresas sin empresa matriz especificada

empresas_pais.loc[
    empresas_pais["Company"] == "Gerry Weber",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Alemania",
    "https://www.bundesanzeiger.de/pub/de/restrukturierungsforum"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Gildan Activewear Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Canadá",
    "https://gildancorp.com/en/contact/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Guess? Inc",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://content.guess.com/GuessUS/GuessWorld/HTML/contact.html"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Gymshark Limited",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://find-and-update.company-information.service.gov.uk/company/08130873"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Heilan Home",
    ["Pais_sede", "Fuente_pais"]
] = [
    "China",
    "https://wemember.heilanhome.com/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Hema",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Países Bajos",
    "https://corporate.hema.com/en/contacteng/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Hermes International",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Francia",
    "https://finance.hermes.com/en/legal-terms/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Hugo Boss AG",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Alemania",
    "https://annualreport.hugoboss.com/2023/management-report/group-profile/business-activities-and-group-structure.html"
]

empresas_pais.loc[
    empresas_pais["Company"] == "JD Sports Fashion plc",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://www.jdplc.com/contact-us/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Jockey International Inc",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.jockey.com/careers-corporate"
]

empresas_pais.loc[
    empresas_pais["Company"] == "John Lewis",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://find-and-update.company-information.service.gov.uk/company/00233462"
]

empresas_pais.loc[
    empresas_pais["Company"] == "K-Way",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Italia",
    "https://www.k-way.com/pages/general-terms-and-conditions-of-sale"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Kathmandu Holdings Ltd",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Nueva Zelanda",
    "https://www.kmdbrands.com/terms-of-use/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Kaufland",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Alemania",
    "https://www.kaufland.com/legal-notice.html"
]

empresas_pais.loc[
    empresas_pais["Company"] == "KiK Textilien und Non-Food GmbH",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Alemania",
    "https://www.kik.de/Rechtliche-Hinweise/Impressum"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Kiabi",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Francia",
    "https://www.kiabi.com/services/donnees-client.html"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Kohl's",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://corporate.kohls.com/contact-us"
]

# Koovs no cuenta con una página corporativa activa.
# Se utiliza un registro empresarial de la compañía operadora.
empresas_pais.loc[
    empresas_pais["Company"] == "Koovs",
    ["Pais_sede", "Fuente_pais"]
] = [
    "India",
    "https://www.zaubacorp.com/company/KOOVS-MARKETING-CONSULTING-PRIVATE-LIMITED/U74140HR2010PTC050747"
]

empresas_pais.loc[
    empresas_pais["Company"] == "LC Waikiki",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Turquía",
    "https://corporate.lcwaikiki.com/en-US/Head-Quarter"
]

empresas_pais.loc[
    empresas_pais["Company"] == "LL Bean",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.llbean.com/llb/shop/518778?page=llbeans-new-maine-headquarters-invites-the-outdoors-in-connecting-employees-to-nature-and-purpose"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Lands' End, Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://investors.landsend.com/investor-faqs"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Levi Strauss & Co.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.levistrauss.com/news/pressroom/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Li-Ning",
    ["Pais_sede", "Fuente_pais"]
] = [
    "China",
    "https://ir.lining.com/en/contact/contact.php"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Lidl Limited",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Alemania",
    "https://careers.lidl.com/lidl-recruiting-portal-terms-of-use"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Longchamp",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Francia",
    "https://www.longchamp.com/gb/en/footer-legal-information.html"
]

empresas_pais.loc[
    empresas_pais["Company"] == "MANGO (Punto Fa, S.L.)",
    ["Pais_sede", "Fuente_pais"]
] = [
    "España",
    "https://www.mangofashiongroup.com/en/w/mango-avanza-con-su-proceso-de-transformaci%C3%B3n-cambia-su-denominaci%C3%B3n-social-y-traslada-su-domicilio-social-al-nuevo-campus-mango-que-se-ampliar%C3%A1-con-dos-nuevos-m%C3%B3dulos"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Marks and Spencer Group plc",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://corporate.marksandspencer.com/contact"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Matalan",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://www.matalan.co.uk/c/customer-services/terms-and-conditions/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Max Mara",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Italia",
    "https://www.maxmarafashiongroup.com/places?lang=en"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Metersbonwe",
    ["Pais_sede", "Fuente_pais"]
] = [
    "China",
    "https://corp.metersbonwe.com/Index/HelpArticle?name=contact_us"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Mexx",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Países Bajos",
    "https://www.mexx.com/en/terms-conditions/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Mizuno Corporation",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Japón",
    "https://corp.mizuno.com/en/about?media=4816"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Moncler",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Italia",
    "https://www.monclergroup.com/wp-content/uploads/2018/07/moncler-inaugurates-its-new-milan-headquarters.pdf"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Mr Price",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Sudáfrica",
    "https://mrpricegroup.com/contact-us/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "New Balance Athletic Shoe, Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://jobs.newbalance.com/global/en/brighton-hq"
]

empresas_pais.loc[
    empresas_pais["Company"] == "New Look Retail Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://www.newlookgroup.com/contact-us"
]

empresas_pais.loc[
    empresas_pais["Company"] == "New Yorker",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Alemania",
    "https://www.newyorker.de/legal/imprint/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Next PLC",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://www.nextplc.co.uk/contact-us"
]

# Validación de la segunda tanda de 38 empresas
print(
    "Empresas todavía pendientes:",
    empresas_pais["Pais_sede"].isna().sum()
)

display(
    empresas_pais[
        empresas_pais["Company"].isin(
            empresas_pendientes.iloc[38:76]["Company"]
        )
    ][
        [
            "Company",
            "Pais_sede",
            "Fuente_pais"
        ]
    ].reset_index(drop=True)
)

Empresas todavía pendientes: 37


,Company,Pais_sede,Fuente_pais
0,Gerry Weber,Alemania,https://www.bundesanzeiger.de/pub/de/restruktu...
1,Gildan Activewear Inc.,Canadá,https://gildancorp.com/en/contact/
2,Guess? Inc,Estados Unidos,https://content.guess.com/GuessUS/GuessWorld/H...
3,Gymshark Limited,Reino Unido,https://find-and-update.company-information.se...
4,Heilan Home,China,https://wemember.heilanhome.com/
5,Hema,Países Bajos,https://corporate.hema.com/en/contacteng/
6,Hermes International,Francia,https://finance.hermes.com/en/legal-terms/
7,Hugo Boss AG,Alemania,https://annualreport.hugoboss.com/2023/managem...
8,JD Sports Fashion plc,Reino Unido,https://www.jdplc.com/contact-us/
9,Jockey International Inc,Estados Unidos,https://www.jockey.com/careers-corporate


In [40]:
# Revisión de la tercera y última tanda de empresas pendientes (77 a 113)

display(
    empresas_pendientes
    .iloc[76:]
    .reset_index(drop=True)
)

,Company,Empresa_oficial,Empresa_matriz,Pais_sede,Fuente_pais,Fuente_empresa_matriz
0,Nordstrom,Nordstrom,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
1,OVS SpA,OVS,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
2,Patagonia Inc.,Patagonia,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
3,Pepe Jeans,Pepe Jeans,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
4,Pimkie,Pimkie,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
5,Puma,Puma,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
6,REI,REI,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
7,REVOLVE,REVOLVE,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
8,Ralph Lauren Corporation,Ralph Lauren,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"
9,River Island,River Island,NaN,<NA>,<NA>,"Fashion Transparency Index 2023, p. 35"


In [41]:
# Incorporación del país de sede para la tercera y última
# tanda de 37 empresas sin empresa matriz especificada

empresas_pais.loc[
    empresas_pais["Company"] == "Nordstrom",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.nordstrom.com/browse/customer-service"
]

empresas_pais.loc[
    empresas_pais["Company"] == "OVS SpA",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Italia",
    "https://www.ovscorporate.it/en/contacts"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Patagonia Inc.",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.patagonia.com/where-we-do-business/owned-and-operated.html"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Pepe Jeans",
    ["Pais_sede", "Fuente_pais"]
] = [
    "España",
    "https://www.pepejeans.com/es/es_es/terms-and-conditions.html"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Pimkie",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Francia",
    "https://www.pimkie.fr/pages/mentions-legales"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Puma",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Alemania",
    "https://www.about.puma.com/en/contact"
]

empresas_pais.loc[
    empresas_pais["Company"] == "REI",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.rei.com/newsroom/article/rei-co-op-releases-2023-impact-report-and-financials-reporting-3-76-billion-in-revenue"
]

empresas_pais.loc[
    empresas_pais["Company"] == "REVOLVE",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://investors.revolve.com/resources/investor-faqs/default.aspx"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Ralph Lauren Corporation",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://corporate.ralphlauren.com/contact"
]

empresas_pais.loc[
    empresas_pais["Company"] == "River Island",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://www.riverisland.com/us/useful-information/associate-of-employee"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Ross Stores",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://investors.rossstores.com/static-files/2906e801-2264-4084-92c2-2677510d62f4"
]

empresas_pais.loc[
    empresas_pais["Company"] == "S.Oliver",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Alemania",
    "https://soliver-group.com/en/impressum"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Salvatore Ferragamo SpA",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Italia",
    "https://group.ferragamo.com/en/investor-relations/ir-contacts"
]

# Se utiliza la política de privacidad accesible de Savage X Fenty.
empresas_pais.loc[
    empresas_pais["Company"] == "Savage X Fenty",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.savagex.com/privacy"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Shein",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Singapur",
    "https://www.sheingroup.com/newsroom/shein-affirms-commitment-to-comply-with-the-eu-s-dsa-following-vlop-designation"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Skechers USA Inc",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://about.skechers.com/constructionhq"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Steve Madden",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://investor.stevemadden.com/resources/investor-faqs"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Superdry plc",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://corporate.superdry.com/governance/legal-statements/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "TOD'S",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Italia",
    "https://www.todsgroup.com/sites/default/files/2023-03/2022%20Annual%20Report.pdf"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Takko Holding GmbH",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Alemania",
    "https://company.takko.com/en-gb/imprint.html"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Target",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://corporate.target.com/investors/annual/2024-annual-report/10-k-report/10-k-cover"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Tchibo GmbH",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Alemania",
    "https://www.tchibo.com/de/en/company/facts-figures"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Ted Baker",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Reino Unido",
    "https://find-and-update.company-information.service.gov.uk/company/02509755/filing-history?page=2"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Tom Ford International LLC",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.tomfordfashion.com/en_US/termsandconditions.html"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Tom Tailor",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Alemania",
    "https://company.tom-tailor.com/en/legal-notice"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Tory Burch LLC",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://www.toryburch.com/content/dam/client-services/privacy-policy/Spain_Privacy_Policy.pdf"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Triumph International",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Suiza",
    "https://www.triumph.com/corporate/hu/modern-slavery-act/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Truworths",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Sudáfrica",
    "https://www.truworths.co.za/occ-public/html/group-at-a-glance.html"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Under Armour",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Estados Unidos",
    "https://about.underarmour.com/en/investors/press-releases--events---presentations.html"
]

empresas_pais.loc[
    empresas_pais["Company"] == "United Arrows",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Japón",
    "https://www.united-arrows.co.jp/en/about/company-info/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "United Colors of Benetton",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Italia",
    "https://www.benettongroup.com/en/the-group/find-us/headquarters/"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Valentino SpA",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Italia",
    "https://www.valentino.com/en-us/experience/corporate-information"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Warehouse Group",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Nueva Zelanda",
    "https://www.thewarehousegroup.co.nz/about-us/about"
]

# Se utiliza el enlace completo de la página oficial del grupo.
empresas_pais.loc[
    empresas_pais["Company"] == "Youngor",
    ["Pais_sede", "Fuente_pais"]
] = [
    "China",
    "https://www.youngor.com/en/about/1092.html"
]

# Se utiliza el aviso legal oficial en alemán de Zalando.
empresas_pais.loc[
    empresas_pais["Company"] == "Zalando SE",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Alemania",
    "https://corporate.zalando.com/de/impressum"
]

empresas_pais.loc[
    empresas_pais["Company"] == "Zeeman",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Países Bajos",
    "https://www.zeeman.com/nl-nl/juridisch/privacy"
]

empresas_pais.loc[
    empresas_pais["Company"] == "lululemon athletica",
    ["Pais_sede", "Fuente_pais"]
] = [
    "Canadá",
    "https://corporate.lululemon.com/investors/investor-resources/investor-contact"
]

# Validación de la tercera y última tanda
print(
    "Empresas todavía pendientes:",
    empresas_pais["Pais_sede"].isna().sum()
)

display(
    empresas_pais[
        empresas_pais["Company"].isin(
            empresas_pendientes.iloc[76:]["Company"]
        )
    ][
        [
            "Company",
            "Pais_sede",
            "Fuente_pais"
        ]
    ].reset_index(drop=True)
)

Empresas todavía pendientes: 0


,Company,Pais_sede,Fuente_pais
0,Nordstrom,Estados Unidos,https://www.nordstrom.com/browse/customer-service
1,OVS SpA,Italia,https://www.ovscorporate.it/en/contacts
2,Patagonia Inc.,Estados Unidos,https://www.patagonia.com/where-we-do-business...
3,Pepe Jeans,España,https://www.pepejeans.com/es/es_es/terms-and-c...
4,Pimkie,Francia,https://www.pimkie.fr/pages/mentions-legales
5,Puma,Alemania,https://www.about.puma.com/en/contact
6,REI,Estados Unidos,https://www.rei.com/newsroom/article/rei-co-op...
7,REVOLVE,Estados Unidos,https://investors.revolve.com/resources/invest...
8,Ralph Lauren Corporation,Estados Unidos,https://corporate.ralphlauren.com/contact
9,River Island,Reino Unido,https://www.riverisland.com/us/useful-informat...


In [42]:
# Validación final de la información de país de sede

print(f"Empresas registradas: {len(empresas_pais)}")
print(f"Empresas únicas: {empresas_pais['Company'].nunique()}")
print(f"Empresas duplicadas: {empresas_pais['Company'].duplicated().sum()}")
print(f"Países pendientes: {empresas_pais['Pais_sede'].isna().sum()}")
print(f"Fuentes pendientes: {empresas_pais['Fuente_pais'].isna().sum()}")

Empresas registradas: 250
Empresas únicas: 250
Empresas duplicadas: 0
Países pendientes: 0
Fuentes pendientes: 0


In [43]:
# Integración del país de sede a la base consolidada

base_final = (
    base_final
    .drop(
        columns=["Pais_sede", "Fuente_pais"],
        errors="ignore"
    )
    .merge(
        empresas_pais[
            ["Company", "Pais_sede", "Fuente_pais"]
        ],
        on="Company",
        how="left",
        validate="many_to_one"
    )
)

print(f"Filas de la base final: {len(base_final)}")
print(f"Países pendientes: {base_final['Pais_sede'].isna().sum()}")
print(f"Fuentes pendientes: {base_final['Fuente_pais'].isna().sum()}")

Filas de la base final: 32500
Países pendientes: 0
Fuentes pendientes: 0


### Diccionarios de variables e indicadores

Como parte del proceso de construcción de la base de datos, se elaboraron dos diccionarios complementarios. El primero documenta el significado y el tipo de dato de las variables que conforman la base consolidada. El segundo describe cada indicador del Fashion Transparency Index 2023, su categoría temática y el tipo de respuesta observado.

Estos diccionarios facilitan la interpretación de la información y servirán como referencia para las etapas posteriores de limpieza y análisis.

In [44]:
# Revisamos las variables que conforman la base final

base_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32500 entries, 0 to 32499
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Answer Page      32500 non-null  object 
 1   Metric           32500 non-null  object 
 2   Company          32500 non-null  object 
 3   Year             32500 non-null  int64  
 4   Value            31244 non-null  object 
 5   Source Page      31000 non-null  object 
 6   Answer ID        10500 non-null  float64
 7   Original Source  10500 non-null  object 
 8   Source Count     10500 non-null  float64
 9   Comments         1998 non-null   object 
 10  ISIN             2714 non-null   object 
 11  Categoria        32500 non-null  object 
 12  Pais_sede        32500 non-null  string 
 13  Fuente_pais      32500 non-null  string 
dtypes: float64(2), int64(1), object(9), string(2)
memory usage: 3.5+ MB


In [45]:
# Construimos el diccionario de variables

diccionario_variables = pd.DataFrame({
    "Variable": base_final.columns,
    "Tipo de dato observado": base_final.dtypes.astype(str).values
})

# Traducimos los tipos de dato para facilitar su interpretación
diccionario_variables["Tipo de dato observado"] = (
    diccionario_variables["Tipo de dato observado"]
    .replace({
        "object": "Texto",
        "string": "Texto",
        "int64": "Entero",
        "float64": "Numérico decimal"
    })
)

# Agregamos la descripción de cada variable
diccionario_variables["Descripción"] = [
    "Enlace de WikiRate correspondiente al registro de la respuesta.",
    "Indicador del Fashion Transparency Index 2023 evaluado para la empresa.",
    "Nombre de la empresa evaluada.",
    "Año al que corresponde la evaluación.",
    "Valor o respuesta reportada para el indicador.",
    "Enlace a la página de WikiRate donde se documenta la fuente de la respuesta.",
    "Identificador único de la respuesta dentro de WikiRate.",
    "Documento o fuente original utilizada como evidencia para la respuesta.",
    "Número de fuentes asociadas con la respuesta.",
    "Comentarios adicionales registrados en WikiRate.",
    "Código ISIN (International Securities Identification Number) de la empresa, cuando se encuentra disponible.",
    "Categoría utilizada para organizar la descarga inicial del indicador: Environment, Social, Governance u Other",
    "País donde se localiza la sede corporativa de la empresa evaluada o, cuando corresponde, de su empresa matriz.",
    "Enlace a la fuente utilizada para verificar el país de sede asignado a la empresa o a su empresa matriz."
]

# Reordenamos las columnas
diccionario_variables = diccionario_variables[
    [
        "Variable",
        "Descripción",
        "Tipo de dato observado"
    ]
]

# Mostramos el diccionario de variables
diccionario_variables

,Variable,Descripción,Tipo de dato observado
0,Answer Page,Enlace de WikiRate correspondiente al registro...,Texto
1,Metric,Indicador del Fashion Transparency Index 2023 ...,Texto
2,Company,Nombre de la empresa evaluada.,Texto
3,Year,Año al que corresponde la evaluación.,Entero
4,Value,Valor o respuesta reportada para el indicador.,Texto
5,Source Page,Enlace a la página de WikiRate donde se docume...,Texto
6,Answer ID,Identificador único de la respuesta dentro de ...,Numérico decimal
7,Original Source,Documento o fuente original utilizada como evi...,Texto
8,Source Count,Número de fuentes asociadas con la respuesta.,Numérico decimal
9,Comments,Comentarios adicionales registrados en WikiRate.,Texto


In [46]:
# Obtenemos la lista única de indicadores de la base final

diccionario_indicadores = pd.DataFrame({
    "Indicador original": sorted(base_final["Metric"].dropna().unique())
})

# Generamos una versión más legible del nombre de cada indicador
diccionario_indicadores["Indicador"] = (
    diccionario_indicadores["Indicador original"]
    .str.replace("Fashion Revolution+", "", regex=False)
)

# Mostramos el número total de indicadores identificados
print(f"Indicadores identificados: {len(diccionario_indicadores)}")

# Mostramos las primeras filas del diccionario
diccionario_indicadores.head()

Indicadores identificados: 130


,Indicador original,Indicador
0,Fashion Revolution+1. Policy & Commitments Score,1. Policy & Commitments Score
1,Fashion Revolution+1.1 Own Operations Policies,1.1 Own Operations Policies
2,Fashion Revolution+1.3 Management Procedures,1.3 Management Procedures
3,Fashion Revolution+1.5 Verified Sustainability...,1.5 Verified Sustainability Report
4,Fashion Revolution+2. Governance Score,2. Governance Score


In [47]:
# Obtenemos la clasificación temática 
# utilizada en los archivos descargados desde WikiRate

categorias_originales = (
    base_final[
        base_final["Categoria"] != "Other"
    ][["Metric", "Categoria"]]
    .drop_duplicates()
)

# Eliminamos el prefijo utilizado por WikiRate
categorias_originales["Metric"] = (
    categorias_originales["Metric"]
    .str.replace("Fashion Revolution+", "", regex=False)
)

# Incorporamos la categoría al diccionario de indicadores
diccionario_indicadores = diccionario_indicadores.merge(
    categorias_originales,
    left_on="Indicador",
    right_on="Metric",
    how="left"
)

# Eliminamos la columna auxiliar
diccionario_indicadores = (
    diccionario_indicadores
    .drop(columns="Metric")
    .rename(columns={"Categoria": "Categoría temática"})
)

# Revisamos cuántos indicadores ya fueron clasificados automáticamente
diccionario_indicadores["Categoría temática"].value_counts(dropna=False)

Categoría temática
NaN            84
Environment    26
Governance     12
Social          8
Name: count, dtype: int64

In [48]:
# Clasificación manual de los indicadores descargados desde Other

categorias_manuales = {
    
    # Governance

    "Supply Chain Policies": "Governance",
    "Supply Chain Policies Align with International Standards": "Governance",
    "Supply Chain Policies Are Contractual": "Governance",
    "Supply Chain Policies in Local Language": "Governance",
    "1. Policy & Commitments Score": "Governance",
    "1.1 Own Operations Policies": "Governance",
    "1.3 Management Procedures": "Governance",
    "1.5 Verified Sustainability Report": "Governance",
    "2.1 Identifies Lead Responsibility for Human Rights & Environmental Issues": "Governance",
    "Accountable Board Member Identified": "Governance",
    "Implementation of Board Level Accountability Described": "Governance",
    "Employee Incentives to Improve Impacts": "Governance",
    "Executive Incentives to Improve Impacts": "Governance",
    "Supplier Incentives to Improve Impacts": "Governance",
    "Responsible Tax Strategy": "Governance",

    # Traceability

    "3. Traceability Score (2023)": "Traceability",
    "3.1 Tier One Factory Disclosure": "Traceability",
    "3.2 Processing Facilities Disclosure": "Traceability",
    "3.3 Raw Materials Suppliers Disclosure": "Traceability",

    # Social

    "Plan for Improving Human Rights Impacts": "Social",
    "Reports on Efforts to Improve Human Rights Impacts": "Social",
    "Describes Human Rights Due Diligence Process": "Social",
    "Approach to Involving Women in Human Rights Due Diligence": "Social",
    "Human Rights Risks Impacts and Violations Identified": "Social",
    "Prevention Mitigation and Remediation of Human Rights Risks": "Social",
    "Human Rights Risk Prevention and Remediation Outcomes Published": "Social",
    "New Production Facility Criteria": "Social",
    "Number or % of off-site worker interviews": "Social",
    "Percentage of Audits including Trade Union Representative": "Social",
    "Summary of Assessment Findings": "Social",
    "Ratings by Named Facilities": "Social",
    "Selected Audit Findings by Named Facilities": "Social",
    "Full Audit Reports by Named Facilities": "Social",
    "Remediation Process": "Social",
    "Affected Stakeholder Engagement in Remediation": "Social",
    "Exit Strategy": "Social",
    "Grievance Mechanism - Direct Employees": "Social",
    "Grievance Mechanism - Supply Chain Workers": "Social",
    "Grievance Mechanism Implementation - Supply Chain Workers": "Social",
    "Grievance Mechanism Disseminated to Supply Chain Workers": "Social",
    "Grievance Mechanism in Supplier Policies": "Social",
    "Grievance Reporting - Supply Chain Workers": "Social",
    "Discloses Approach to Recruitment Fees": "Social",
    "Discloses Data on Modern Slavery Prevalence": "Social",
    "Discloses Approach to Living Wage": "Social",
    "Discloses Strategy to Achieving Living Wage": "Social",
    "Discloses Progress toward the payment of a Living Wage to workers in the supply chain": "Social",
    "Discloses Living Wage Estimates Used for Benchmarking": "Social",
    "Discloses Percentage of Workers Receiving Wage Payments Digitally": "Social",
    "Publishes Percentage of Workers Paid Above Minimum Wage": "Social",
    "Publishes Percentage or Number of Workers Earning a Living Wage": "Social",
    "Protects Labour Costs in Price Negotiations": "Social",
    "Discloses Quantity of Orders with Labor Cost Protection": "Social",
    "Policy to Pay Supplier Within 60 Days": "Social",
    "Discloses Quantity of Orders Changed After Original Agreement": "Social",
    "Publishes Supplier Feedback on Purchasing Practices": "Social",
    "Discloses number or % of supplier facilities that have independent, democratically elected trade unions": "Social",
    "Discloses number or % of Workers covered by Collective Bargaining Agreements": "Social",
    "Publishes Gender Pay Gap": "Social",
    "Publishes Sex-disaggregated Job Distribution": "Social",
    "Publishes Gender-based Labour Violations Data": "Social",
    "Discloses Gender Equality Actions in Supplier Facilities": "Social",
    "Publishes Ethnicity Pay Gap": "Social",
    "Publishes Race-disaggregated Job Distribution": "Social",

    # Environment

    "Plan for Improving Environmental Impacts": "Environment",
    "Reports on Efforts to Improve Environmental Impacts": "Environment",
    "Sustainable Materials Strategy": "Environment",
    "Discloses Progress on Sustainable Materials Strategy": "Environment",
    "Targets to Reduce Virgin Plastics": "Environment",
    "Discloses Progress to Reducing Virgin Plastics": "Environment",
    "Minimizing Impact of Microfibres": "Environment",
    "Discloses Quantity of Products Produced": "Environment",
    "Commitment to Degrowth": "Environment",
    "Discloses Quantity of Products Destroyed": "Environment",
    "Offers Take-back Schemes": "Environment",
    "Offers Repair Services": "Environment",
    "Discloses Evidence of Developing Circular Solutions": "Environment",
    "Discloses % of Circular Products": "Environment",
    "Commitment to Eliminate Hazardous Chemicals": "Environment",
    "Discloses Progress to Eliminate Hazardous Chemicals": "Environment",
    "Science Based Targets": "Environment",
    "Environmental Profit and Loss Statement": "Environment",
    "Zero Deforestation Commitment": "Environment",
    "Implementation of Regenerative Farming Practices": "Environment",
    "Discloses Renewable Energy Use": "Environment"
}

In [49]:
# Completamos las categorías faltantes

diccionario_indicadores["Categoría temática"] = (
    diccionario_indicadores["Categoría temática"]
    .fillna(
        diccionario_indicadores["Indicador"].map(categorias_manuales)
    )
)

In [50]:
# Verificamos que todos los indicadores tengan categoría temática

print(
    diccionario_indicadores["Categoría temática"]
    .value_counts(dropna=False)
)

print(
    "\nIndicadores sin categoría:",
    diccionario_indicadores["Categoría temática"].isna().sum()
)

Categoría temática
Social          53
Environment     47
Governance      26
Traceability     4
Name: count, dtype: int64

Indicadores sin categoría: 0


In [51]:
# Identificamos el tipo de respuesta observado para cada indicador

tipos_respuesta = []

for indicador in diccionario_indicadores["Indicador original"]:

    # Valores observados para el indicador
    valores = (
        base_final.loc[
            base_final["Metric"] == indicador,
            "Value"
        ]
        .dropna()
        .astype(str)
    )

    # Clasificamos automáticamente el tipo de respuesta
    if valores.isin(["Yes", "No"]).all():
        tipo = "Sí / No"

    elif valores.str.fullmatch(r"-?\d+(\.\d+)?").all():
        tipo = "Numérico"

    elif valores.str.contains("%").any():
        tipo = "Porcentaje"

    elif valores.str.contains(",").any():
        tipo = "Lista de elementos"

    else:
        tipo = "Texto"

    tipos_respuesta.append(tipo)

# Agregamos la columna al diccionario
diccionario_indicadores["Tipo de respuesta"] = tipos_respuesta

In [52]:
# Imprimimos las primeras 20 filas

diccionario_indicadores[
    ["Indicador", "Tipo de respuesta"]
].head(20)

,Indicador,Tipo de respuesta
0,1. Policy & Commitments Score,Numérico
1,1.1 Own Operations Policies,Lista de elementos
2,1.3 Management Procedures,Lista de elementos
3,1.5 Verified Sustainability Report,Sí / No
4,2. Governance Score,Numérico
5,2.1 Identifies Lead Responsibility for Human R...,Texto
6,3. Traceability Score (2023),Numérico
7,3.1 Tier One Factory Disclosure,Porcentaje
8,3.2 Processing Facilities Disclosure,Porcentaje
9,3.3 Raw Materials Suppliers Disclosure,Porcentaje


In [53]:
# Revisamos cuántos indicadores hay por tipo de respuesta

diccionario_indicadores["Tipo de respuesta"].value_counts()

Tipo de respuesta
Sí / No               110
Lista de elementos     10
Numérico                6
Porcentaje              3
Texto                   1
Name: count, dtype: int64

In [54]:
# Revisamos los indicadores clasificados como "Texto"

diccionario_indicadores.loc[
    diccionario_indicadores["Tipo de respuesta"] == "Texto",
    ["Indicador", "Tipo de respuesta"]
]

,Indicador,Tipo de respuesta
5,2.1 Identifies Lead Responsibility for Human R...,Texto


In [55]:
# Identificamos los indicadores numéricos que corresponden a puntajes

indicadores_puntaje = [
    "1. Policy & Commitments Score",
    "2. Governance Score",
    "3. Traceability Score (2023)",
    "4. Know, Show & Fix Score",
    "5. Spotlight Issues Score (2023)",
    "Fashion Transparency Index 2023"
]

# Reemplazamos su tipo de respuesta
diccionario_indicadores.loc[
    diccionario_indicadores["Indicador"].isin(indicadores_puntaje),
    "Tipo de respuesta"
] = "Puntaje"

In [56]:
# Agregamos una breve descripción de cada indicador

significados = {

    "1. Policy & Commitments Score": "Puntaje de transparencia sobre políticas y compromisos de sostenibilidad.",
    "1.1 Own Operations Policies": "Políticas aplicables a las operaciones propias de la empresa.",
    "1.3 Management Procedures": "Procedimientos de gestión para implementar las políticas de sostenibilidad.",
    "1.5 Verified Sustainability Report": "Publicación de un informe de sostenibilidad verificado externamente.",
    "2. Governance Score": "Puntaje de transparencia sobre gobernanza y rendición de cuentas.",
    "2.1 Identifies Lead Responsibility for Human Rights & Environmental Issues": "Persona o área responsable de los temas de derechos humanos y medio ambiente.",
    "3. Traceability Score (2023)": "Puntaje de transparencia sobre trazabilidad en la cadena de suministro.",
    "3.1 Tier One Factory Disclosure": "Divulgación de las fábricas de primer nivel de la cadena de suministro.",
    "3.2 Processing Facilities Disclosure": "Divulgación de las instalaciones donde se procesan los materiales.",
    "3.3 Raw Materials Suppliers Disclosure": "Divulgación de los proveedores de materias primas.",
    "4. Know, Show & Fix Score": "Puntaje sobre identificación, divulgación y corrección de impactos.",
    "5. Spotlight Issues Score (2023)": "Puntaje sobre transparencia en temas prioritarios de sostenibilidad.",
    "Accountable Board Member Identified": "Identificación de un miembro del consejo responsable de sostenibilidad.",
    "Affected Stakeholder Engagement in Remediation": "Participación de las partes afectadas en los procesos de remediación.",
    "Approach to Defining Sustainable Materials": "Criterios utilizados para definir materiales sostenibles.",
    "Approach to Involving Women in Human Rights Due Diligence": "Acciones para incluir a las mujeres en la debida diligencia de derechos humanos.",
    "Commitment to Degrowth": "Compromiso con estrategias de reducción del crecimiento productivo.",
    "Commitment to Eliminate Hazardous Chemicals": "Compromiso para eliminar sustancias químicas peligrosas.",
    "Decarbonisation Commitment": "Compromiso para reducir las emisiones de carbono.",
    "Decarbonisation Progress": "Reporte de avances en las metas de descarbonización.",
    "Describes Environmental Due Diligence Process": "Descripción del proceso de debida diligencia ambiental.",
    "Describes Human Rights Due Diligence Process": "Descripción del proceso de debida diligencia en derechos humanos.",
    "Discloses % of Circular Products": "Divulgación del porcentaje de productos circulares.",
    "Discloses Absolute Energy Reduction": "Divulgación de la reducción absoluta en el consumo de energía.",
    "Discloses Annual Investment in Decarbonisation": "Divulgación de la inversión anual destinada a la descarbonización.",
    "Discloses Approach to Living Wage": "Divulgación del enfoque para promover salarios dignos.",
    "Discloses Approach to Recruitment Fees": "Divulgación del enfoque para gestionar las cuotas de contratación.",
    "Discloses Breakdown of Reuse/Recyling of Pre-consumer Waste": "Divulgación del destino de los residuos previos al consumo.",
    "Discloses Coal Use": "Divulgación del uso de carbón en las operaciones.",
    "Discloses Content of Scope 1, 2 and 3 Emissions": "Divulgación de emisiones de alcance 1, 2 y 3.",
    "Discloses Data on Modern Slavery Prevalence": "Divulgación de información sobre esclavitud moderna.",
    "Discloses Efforts to Invest in Supply Chain Workers": "Divulgación de inversiones dirigidas a trabajadores de la cadena de suministro.",
    "Discloses Evidence of Developing Circular Solutions": "Divulgación de acciones para desarrollar soluciones circulares.",
    "Discloses Free on Board (FOB) Price Changes (COVID-19 Response)": "Divulgación de cambios en precios FOB durante la respuesta a COVID-19.",
    "Discloses Gender Equality Actions in Supplier Facilities": "Divulgación de acciones para promover la igualdad de género en proveedores.",
    "Discloses Living Wage Estimates Used for Benchmarking": "Divulgación de estimaciones de salario digno utilizadas como referencia.",
    "Discloses Number of Collective Bargaining Agreements Providing Wages Above Legal Minimum": "Divulgación de convenios colectivos con salarios superiores al mínimo legal.",
    "Discloses Number of Workers Affected by Recruitment Fees": "Divulgación del número de trabajadores afectados por cuotas de contratación.",
    "Discloses Percentage of Workers Paid By Piece Rate": "Divulgación del porcentaje de trabajadores remunerados por destajo.",
    "Discloses Percentage of Workers Receiving Wage Payments Digitally": "Divulgación del porcentaje de trabajadores que reciben pagos digitales.",
    "Discloses Policy on Up-Front Supplier Payments": "Divulgación de la política de pagos anticipados a proveedores.",
    "Discloses Prevalance of Collective Bargaining Violations": "Divulgación de casos de incumplimiento de negociación colectiva.",
    "Discloses Progress on Sustainable Materials Strategy": "Divulgación de avances en la estrategia de materiales sostenibles.",
    "Discloses Progress to Eliminate Hazardous Chemicals": "Divulgación de avances para eliminar sustancias químicas peligrosas.",
    "Discloses Progress to Reducing Textiles Derived from Virgin Fossil Fuels": "Divulgación de avances para reducir textiles derivados de combustibles fósiles vírgenes.",
    "Discloses Progress to Reducing Virgin Plastics": "Divulgación de avances para reducir el uso de plásticos vírgenes.",
    "Discloses Progress toward the payment of a Living Wage to workers in the supply chain": "Divulgación de avances en el pago de salarios dignos en la cadena de suministro.",
    "Discloses Proportion of Workers Paid Minimum Wage": "Divulgación de la proporción de trabajadores que reciben el salario mínimo.",
    "Discloses Quantity of Orders Changed After Original Agreement": "Divulgación de la cantidad de pedidos modificados después del acuerdo inicial.",
    "Discloses Quantity of Orders with Labor Cost Protection": "Divulgación de pedidos que protegen los costos laborales.",
    "Discloses Quantity of Post-production Waste Generated": "Divulgación de la cantidad de residuos generados después de la producción.",
    "Discloses Quantity of Pre-Production Waste Generated": "Divulgación de la cantidad de residuos generados antes de la producción.",
    "Discloses Quantity of Products Destroyed": "Divulgación de la cantidad de productos destruidos.",
    "Discloses Quantity of Products Produced": "Divulgación de la cantidad de productos fabricados.",
    "Discloses Renewable Energy Use": "Divulgación del uso de energías renovables.",
    "Discloses Sourced Fibre Breakdown": "Divulgación de la composición de las fibras utilizadas.",
    "Discloses Strategy to Achieving Living Wage": "Divulgación de la estrategia para alcanzar salarios dignos.",
    "Discloses Take-back Scheme Outcomes": "Divulgación de los resultados de los programas de devolución de productos.",
    "Discloses Time Taken to Pay Purchase Orders": "Divulgación del tiempo requerido para pagar órdenes de compra.",
    "Discloses Water Use": "Divulgación del consumo de agua.",
    "Discloses Water-Related Risk Assessment Process": "Divulgación del proceso de evaluación de riesgos relacionados con el agua.",
    "Discloses number or % of Workers covered by Collective Bargaining Agreements": "Divulgación del número o porcentaje de trabajadores cubiertos por convenios colectivos.",
    "Discloses number or % of supplier facilities that have independent, democratically elected trade unions": "Divulgación del número o porcentaje de proveedores con sindicatos independientes.",
    "Employee Incentives to Improve Impacts": "Incentivos para que los empleados mejoren los impactos de sostenibilidad.",
    "Environmental Profit and Loss Statement": "Publicación de un estado de ganancias y pérdidas ambientales.",
    "Environmental Risk Prevention and Remediation Outcomes Published": "Publicación de resultados sobre prevención y remediación de riesgos ambientales.",
    "Environmental Risks Impacts and Violations Identified": "Identificación de riesgos, impactos e incumplimientos ambientales.",
    "Executive Incentives to Improve Impacts": "Incentivos para directivos vinculados a mejoras en sostenibilidad.",
    "Executive Pay Linked to Environmental and Social Targets": "Vinculación de la remuneración directiva con objetivos ambientales y sociales.",
    "Exit Strategy": "Divulgación de la estrategia para finalizar relaciones con proveedores.",
    "Fashion Transparency Index 2023": "Puntaje general obtenido en el Fashion Transparency Index 2023.",
    "Full Audit Reports by Named Facilities": "Publicación de informes completos de auditoría por instalación.",
    "Grievance Mechanism - Direct Employees": "Existencia de mecanismos de quejas para empleados directos.",
    "Grievance Mechanism - Supply Chain Workers": "Existencia de mecanismos de quejas para trabajadores de la cadena de suministro.",
    "Grievance Mechanism Disseminated to Supply Chain Workers": "Difusión de los mecanismos de quejas entre trabajadores de la cadena de suministro.",
    "Grievance Mechanism Implementation - Supply Chain Workers": "Implementación de mecanismos de quejas para trabajadores de la cadena de suministro.",
    "Grievance Mechanism in Supplier Policies": "Inclusión de mecanismos de quejas en las políticas para proveedores.",
    "Grievance Reporting - Supply Chain Workers": "Divulgación de reportes derivados de mecanismos de quejas.",
    "Human Rights Risk Prevention and Remediation Outcomes Published": "Publicación de resultados sobre prevención y remediación de riesgos en derechos humanos.",
    "Human Rights Risks Impacts and Violations Identified": "Identificación de riesgos, impactos e incumplimientos en derechos humanos.",
    "Implementation of Board Level Accountability Described": "Descripción de cómo el consejo asume responsabilidades en sostenibilidad.",
    "Implementation of Regenerative Farming Practices": "Divulgación de prácticas de agricultura regenerativa.",
    "Minimizing Impact of Microfibres": "Acciones para reducir el impacto de las microfibras.",
    "New Production Facility Criteria": "Criterios utilizados para seleccionar nuevas instalaciones de producción.",
    "Number or % of off-site worker interviews": "Divulgación del número o porcentaje de entrevistas realizadas fuera del lugar de trabajo.",
    "Offers Clothing Longevity Business Models": "Oferta de modelos de negocio que prolongan la vida útil de las prendas.",
    "Offers Repair Services": "Oferta de servicios de reparación de prendas.",
    "Offers Take-back Schemes": "Oferta de programas para recuperar productos usados.",
    "Percentage of Audits including Trade Union Representative": "Divulgación del porcentaje de auditorías con participación sindical.",
    "Plan for Improving Environmental Impacts": "Plan para reducir los impactos ambientales.",
    "Reports on Efforts to Improve Environmental Impacts": "Reporte de acciones para mejorar los impactos ambientales.",
    "Reports on Efforts to Improve Human Rights Impacts": "Reporte de acciones para mejorar los impactos en derechos humanos.",
    "Reports on Minimum Wage Paid for Daily / Piece Rate Workers": "Reporte sobre el salario mínimo pagado a trabajadores por día o destajo.",
    "Responsible Tax Strategy": "Divulgación de una estrategia fiscal responsable.",
    "Science Based Targets": "Adopción de metas climáticas alineadas con la ciencia.",
    "Scope, Process and Accreditation for Environmental Audits": "Divulgación del alcance, proceso y acreditación de las auditorías ambientales.",
    "Selected Audit Findings by Named Facilities": "Divulgación de hallazgos de auditorías por instalación.",
    "Stakeholder Engagement in Environmental Due Diligence": "Participación de las partes interesadas en la debida diligencia ambiental.",
    "Stakeholder Engagement in Human Rights Due Diligence": "Participación de las partes interesadas en la debida diligencia en derechos humanos.",
    "Summary of Assessment Findings": "Resumen de los resultados de las evaluaciones realizadas.",
    "Supplier Incentives to Improve Impacts": "Incentivos para que los proveedores mejoren sus impactos.",
    "Supply Chain Policies": "Políticas aplicables a la cadena de suministro.",
    "Supply Chain Policies Align with International Standards": "Alineación de las políticas con estándares internacionales.",
    "Supply Chain Policies Are Contractual": "Incorporación de las políticas en contratos con proveedores.",
    "Supply Chain Policies in Local Language": "Disponibilidad de las políticas en el idioma local.",
    "Sustainable Materials Strategy": "Estrategia para incrementar el uso de materiales sostenibles.",
    "Targets to Reduce Textiles Derived from Virgin Fossil Fuels": "Metas para reducir textiles derivados de combustibles fósiles vírgenes.",
    "Targets to Reduce Virgin Plastics": "Metas para reducir el uso de plásticos vírgenes.",
    "Worker Representation on Board": "Representación de los trabajadores en el consejo de administración.",
    "Zero Deforestation Commitment": "Compromiso para eliminar la deforestación en la cadena de suministro.",
    "Zero Deforestation Progress": "Reporte de avances hacia la eliminación de la deforestación.",
    "Plan for Improving Human Rights Impacts": "Plan para reducir los impactos en derechos humanos.",
    "Policy to Pay Supplier Within 60 Days": "Política para pagar a los proveedores en un plazo máximo de 60 días.",
    "Prevention Mitigation and Remediation of Environmental Risks": "Acciones para prevenir, mitigar y remediar riesgos ambientales.",
    "Prevention Mitigation and Remediation of Human Rights Risks": "Acciones para prevenir, mitigar y remediar riesgos en derechos humanos.",
    "Protects Labour Costs in Price Negotiations": "Protección de los costos laborales durante la negociación de precios.",
    "Publishes Ethnicity Pay Gap": "Publicación de la brecha salarial por origen étnico.",
    "Publishes Gender Pay Gap": "Publicación de la brecha salarial de género.",
    "Publishes Gender-based Labour Violations Data": "Publicación de datos sobre violaciones laborales por motivo de género.",
    "Publishes Percentage of Workers Paid Above Minimum Wage": "Publicación del porcentaje de trabajadores que reciben un salario superior al mínimo.",
    "Publishes Percentage or Number of Workers Earning a Living Wage": "Publicación del número o porcentaje de trabajadores que reciben un salario digno.",
    "Publishes Race-disaggregated Job Distribution": "Publicación de la distribución de puestos desagregada por raza.",
    "Publishes Racial Equality Actions": "Publicación de acciones para promover la igualdad racial.",
    "Publishes Responsible Purchasing Code of Conduct": "Publicación del código de conducta para compras responsables.",
    "Publishes Sex-disaggregated Job Distribution": "Publicación de la distribución de puestos desagregada por sexo.",
    "Publishes Standard Supplier Agreement Template": "Publicación del modelo estándar de contrato con proveedores.",
    "Publishes Supplier Feedback on Purchasing Practices": "Publicación de la retroalimentación de proveedores sobre las prácticas de compra.",
    "Publishes Supplier Wastewater Test Results": "Publicación de resultados de pruebas de aguas residuales de proveedores.",
    "Ratings by Named Facilities": "Publicación de calificaciones por instalación.",
    "Remediation Process": "Descripción del proceso de remediación aplicado por la empresa."
}

# Incorporamos el significado al diccionario
diccionario_indicadores["Significado"] = (
    diccionario_indicadores["Indicador"]
    .map(significados)
)

# Reordenamos las columnas
diccionario_indicadores = diccionario_indicadores[
    [
        "Indicador original",
        "Indicador",
        "Significado",
        "Categoría temática",
        "Tipo de respuesta"
    ]
]

# Mostramos el diccionario
diccionario_indicadores

,Indicador original,Indicador,Significado,Categoría temática,Tipo de respuesta
0,Fashion Revolution+1. Policy & Commitments Score,1. Policy & Commitments Score,Puntaje de transparencia sobre políticas y com...,Governance,Puntaje
1,Fashion Revolution+1.1 Own Operations Policies,1.1 Own Operations Policies,Políticas aplicables a las operaciones propias...,Governance,Lista de elementos
2,Fashion Revolution+1.3 Management Procedures,1.3 Management Procedures,Procedimientos de gestión para implementar las...,Governance,Lista de elementos
3,Fashion Revolution+1.5 Verified Sustainability...,1.5 Verified Sustainability Report,Publicación de un informe de sostenibilidad ve...,Governance,Sí / No
4,Fashion Revolution+2. Governance Score,2. Governance Score,Puntaje de transparencia sobre gobernanza y re...,Governance,Puntaje
...,...,...,...,...,...
125,Fashion Revolution+Targets to Reduce Textiles ...,Targets to Reduce Textiles Derived from Virgin...,Metas para reducir textiles derivados de combu...,Environment,Sí / No
126,Fashion Revolution+Targets to Reduce Virgin Pl...,Targets to Reduce Virgin Plastics,Metas para reducir el uso de plásticos vírgenes.,Environment,Sí / No
127,Fashion Revolution+Worker Representation on Board,Worker Representation on Board,Representación de los trabajadores en el conse...,Governance,Sí / No
128,Fashion Revolution+Zero Deforestation Commitment,Zero Deforestation Commitment,Compromiso para eliminar la deforestación en l...,Environment,Sí / No


In [57]:
# Verificamos que todos los indicadores tengan significado

print("Indicadores definidos:", len(significados))
print("Significados faltantes:", diccionario_indicadores["Significado"].isna().sum())

Indicadores definidos: 130
Significados faltantes: 0


### Validación final de la base de datos y los diccionarios

In [58]:
# VALIDACIÓN FINAL DE LA BASE DE DATOS

# Mostramos el número de registros y variables
print("Número de registros:", len(base_final))
print("Número de variables:", base_final.shape[1])


# Verificamos si existen valores faltantes
print("\nValores faltantes por variable:")

print(
    base_final
    .isna()
    .sum()
)


# Verificamos si existen registros duplicados
print("\nRegistros duplicados:")

print(
    base_final
    .duplicated()
    .sum()
)


# Mostramos el tipo de dato de cada variable
print("\nTipo de dato de cada variable:")

print(
    base_final
    .dtypes
)

Número de registros: 32500
Número de variables: 14

Valores faltantes por variable:
Answer Page            0
Metric                 0
Company                0
Year                   0
Value               1256
Source Page         1500
Answer ID          22000
Original Source    22000
Source Count       22000
Comments           30502
ISIN               29786
Categoria              0
Pais_sede              0
Fuente_pais            0
dtype: int64

Registros duplicados:
0

Tipo de dato de cada variable:
Answer Page                object
Metric                     object
Company                    object
Year                        int64
Value                      object
Source Page                object
Answer ID                 float64
Original Source            object
Source Count              float64
Comments                   object
ISIN                       object
Categoria                  object
Pais_sede          string[python]
Fuente_pais        string[python]
dtype: object


**Observación**

Los valores faltantes identificados corresponden a variables cuyo contenido no está disponible para todos los registros en la fuente original de Fashion Revolution (por ejemplo, *Comments*, *ISIN*, *Answer ID*, *Original Source* y *Source Count*), así como se identifica ausencia de algunos valores en variables no binarias. Por otro lado, la variable **Categoria** conserva la clasificación utilizada durante la organización de las descargas: **Environment**, **Social**, **Governance** y **Other**. Posteriormente, la clasificación temática definitiva de cada indicador se documenta en el diccionario de indicadores.

In [59]:
# VALIDACIÓN FINAL DE LOS DICCIONARIOS

# Verificamos que todas las variables de la base estén documentadas
print("Variables en la base:")

print(
    len(base_final.columns)
)

print("\nVariables en el diccionario:")

print(
    len(diccionario_variables)
)


# Comparamos ambos conjuntos de variables
print("\n¿Las variables coinciden?")

print(
    set(base_final.columns)
    ==
    set(diccionario_variables["Variable"])
)


# Verificamos que todos los indicadores estén documentados
print("\nIndicadores en la base:")

print(
    base_final["Metric"]
    .nunique()
)

print("\nIndicadores en el diccionario:")

print(
    len(diccionario_indicadores)
)


# Comparamos ambos conjuntos de indicadores
print("\n¿Los indicadores coinciden?")

print(
    set(
        base_final["Metric"]
        .str.replace(
            "Fashion Revolution+",
            "",
            regex=False
        )
    )
    ==
    set(diccionario_indicadores["Indicador"])
)


# Verificamos que no existan significados faltantes
print("\nSignificados faltantes:")

print(
    diccionario_indicadores["Significado"]
    .isna()
    .sum()
)


# Verificamos que no existan categorías temáticas faltantes
print("\nCategorías temáticas faltantes:")

print(
    diccionario_indicadores["Categoría temática"]
    .isna()
    .sum()
)


# Verificamos que no existan tipos de respuesta faltantes
print("\nTipos de respuesta faltantes:")

print(
    diccionario_indicadores["Tipo de respuesta"]
    .isna()
    .sum()
)


# Verificamos que no existan indicadores duplicados
print("\nIndicadores duplicados:")

print(
    diccionario_indicadores["Indicador"]
    .duplicated()
    .sum()
)


# Mostramos la distribución de categorías temáticas
print("\nCategorías temáticas:")

print(
    diccionario_indicadores["Categoría temática"]
    .value_counts()
)


# Mostramos la distribución de tipos de respuesta
print("\nTipos de respuesta:")

print(
    diccionario_indicadores["Tipo de respuesta"]
    .value_counts()
)

Variables en la base:
14

Variables en el diccionario:
14

¿Las variables coinciden?
True

Indicadores en la base:
130

Indicadores en el diccionario:
130

¿Los indicadores coinciden?
True

Significados faltantes:
0

Categorías temáticas faltantes:
0

Tipos de respuesta faltantes:
0

Indicadores duplicados:
0

Categorías temáticas:
Categoría temática
Social          53
Environment     47
Governance      26
Traceability     4
Name: count, dtype: int64

Tipos de respuesta:
Tipo de respuesta
Sí / No               110
Lista de elementos     10
Puntaje                 6
Porcentaje              3
Texto                   1
Name: count, dtype: int64


### Exportación de la base de datos y los diccionarios

In [60]:
# Exportación de la base de datos y los diccionarios

# Exportamos la base de datos integrada
base_final.to_csv(
    ruta_procesados / "base_integrada.csv",
    index=False
)


# Exportamos el diccionario de variables
diccionario_variables.to_excel(
    ruta_procesados / "diccionario_variables.xlsx",
    index=False
)


# Exportamos el diccionario de indicadores
diccionario_indicadores.to_excel(
    ruta_procesados / "diccionario_indicadores.xlsx",
    index=False
)


# Confirmamos que la exportación finalizó correctamente
print("Archivos exportados correctamente.")

Archivos exportados correctamente.
